1. 共用參數設定


In [ ]:
'''
Measure the elapsed time of every code cell, including cells that raise errors.
記錄每個 Code Cell 的執行時間，即使發生錯誤也顯示已耗費的時間。
'''
from time import perf_counter as _cell_perf_counter
from IPython import get_ipython as _cell_get_ipython

_CELL_TIMER_SHELL = _cell_get_ipython()
if _CELL_TIMER_SHELL is None:
    raise RuntimeError('請在 Jupyter Notebook 中執行本教材。')
_CELL_TIMER_STATE = {'started': _cell_perf_counter()}

def _cell_timer_begin(_info):
    _CELL_TIMER_STATE['started'] = _cell_perf_counter()

def _cell_timer_end(result):
    started = _CELL_TIMER_STATE.pop('started', None)
    if started is None:
        return
    minutes, seconds = divmod(_cell_perf_counter() - started, 60)
    elapsed = f'{int(minutes)} 分 {seconds:05.2f} 秒' if minutes else f'{seconds:.2f} 秒'
    status = '執行失敗' if result.error_before_exec or result.error_in_exec else '執行完成'
    print(f'⏱ 本格{status}，耗時：{elapsed}')

'''
Replace existing timers when the first cell is executed again.
重新執行第一格時，先取消舊計時器，避免重複顯示耗時。
'''
for _event_name, _callback in globals().get('_CELL_TIMER_CALLBACKS', ()):
    try:
        _CELL_TIMER_SHELL.events.unregister(_event_name, _callback)
    except ValueError:
        pass
_CELL_TIMER_CALLBACKS = (
    ('pre_run_cell', _cell_timer_begin),
    ('post_run_cell', _cell_timer_end),
)
for _event_name, _callback in _CELL_TIMER_CALLBACKS:
    _CELL_TIMER_SHELL.events.register(_event_name, _callback)

'''
Import standard-library tools for editable paths and local service settings.
匯入設定路徑與本機服務參數所需的 Python 內建工具。
'''
import os
from pathlib import Path
ROOT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.home() / 'ai_course'
ROOT = ROOT.expanduser().resolve()  # Ubuntu 可改成 Path('/home/your_user/ai_course')
SCRATCH_ROOT = Path('/tmp') if Path('/kaggle/working').is_dir() else ROOT / 'cache'
SCRATCH_ROOT = SCRATCH_ROOT.expanduser().resolve()  # 可改成其他儲存裝置或模型磁碟
ROOT.mkdir(parents=True, exist_ok=True)
SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)

COMFY_DIR = ROOT / 'ComfyUI'
GPU_COUNT = 2  # Kaggle T4 ×2；可在 NVIDIA Linux 主機調整
MODEL_GPU_ID = 0  # 擴散模型的主要 GPU
AUX_GPU_ID = 1  # 文字編碼器、VAE 與 Upscaler

'''
Release ComfyUI models and node caches after each completed video workflow.
每支影片生成完成後，清理 ComfyUI 模型與節點快取，但不停止服務。
'''
RELEASE_AFTER_GENERATION = False

'''
Stop ComfyUI after each finished video so Kaggle RAM returns to a clean state.
每支影片完成後停止 ComfyUI，讓 Kaggle RAM 回到乾淨狀態；下次生成前再自動啟動。
'''
STOP_COMFY_AFTER_GENERATION = True


'''
Share the cleanup policy with notebook and API engine processes.
讓 Notebook 與 API 生成引擎採用相同的清理設定。
'''
os.environ['LTX25_RELEASE_AFTER_GENERATION'] = '1' if RELEASE_AFTER_GENERATION else '0'
os.environ['LTX25_STOP_AFTER_GENERATION'] = '1' if STOP_COMFY_AFTER_GENERATION else '0'

'''
Choose delivery policy: exact user dimensions or the larger native model video.
選擇交付尺寸：符合使用者設定，或明確選擇模型原始較大尺寸。
'''
OUTPUT_POLICY = 'exact_final'  # 'exact_final' or 'native'; x2 source remains auditable

'''
Select the model and workflow storage locations.
設定 LTX 模型下載、Workflow 註冊表與引擎檔案的路徑。
'''
SCRATCH_DIR = SCRATCH_ROOT / 'ltx25'
MODEL_STORE = SCRATCH_DIR / 'ltx25_models'
FLOW_REGISTRY_PATH = ROOT / 'ltx25_local_flows.json'
COMFY_LOG = ROOT / 'comfyui_ltx25.log'
COMFY_PORT = 8188  # 8188、8189 等未使用的 Port
COMFY_REF = 'v0.45.0'

'''
Choose the model cache folder and apply it before downloads.
設定模型下載快取位置，並在下載前套用。
'''
HF_CACHE_DIR = SCRATCH_DIR / 'huggingface'
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_XET_CACHE'] = str(HF_CACHE_DIR / 'xet')
NODE_CACHE_MODE = 'none'  # 課堂穩定版：停用節點快取，降低 RAM 累積風險
GPU_VRAM_RESERVE_GIB = 1  # 1、2，單位 GiB

'''
Require enough available host RAM before loading a large video workflow.
設定生成前主機／容器 RAM 的最低安全門檻。
'''
LTX_MIN_AVAILABLE_RAM_GIB = 8.0  # 例如 6.0、8.0 GiB
MODEL_TIER = 'q4km'  # 此版固定；切換需另備相容權重與 Workflow
FPS = 24
VAE_TILE_SIZE, VAE_TILE_OVERLAP = 256, 32
VAE_TEMPORAL_SIZE, VAE_TEMPORAL_OVERLAP = 16, 4

'''
Select the optional remote FIFO API port; change only if the port conflicts.
指定選用 FIFO 遠端 API 的本機連接埠；只有發生 Port 衝突時才需要修改。
'''
REMOTE_PORT = 8765  # 例如 8765、8766，範圍 1024～65535
REMOTE_SERVICE_ID = '06_LTX-2.5_影片生成_Kaggle'
REMOTE_ROOT = ROOT / 'remote_api_jobs' / REMOTE_SERVICE_ID

'''
Limit how many remote jobs wait for the single GPU worker.
設定 FIFO 最多等待的工作數量；超出時回傳 HTTP 429，client 會延遲重試。
'''
REMOTE_QUEUE_CAPACITY = 32  # 建議 12～32，依課堂人數調整
REMOTE_UPLOAD_CONCURRENCY = 2  # 建議 1～2
REMOTE_RESULT_TTL_HOURS = 24
REMOTE_RESULT_LIMIT_GIB = 8
REMOTE_MIN_FREE_GIB = 2

'''
Mark exceptionally long remote inference as stalled, without forcibly starting a second GPU job.
設定遠端推論超時警戒值；卡住時不強行啟動第二筆 GPU 推論。
'''
REMOTE_STALL_HOURS = float('inf')  # 不限制單次生成時長，也不將執行中的工作誤判為停滯
LTX_OUTPUT_DIR = ROOT / 'ltx25_api_outputs'
os.environ['LTX25_MODEL_GPU_DEVICE'] = f'cuda:{MODEL_GPU_ID}'
os.environ['LTX25_AUX_GPU_DEVICE'] = f'cuda:{AUX_GPU_ID}'
os.environ['COMFY_BASE'] = f'http://127.0.0.1:{COMFY_PORT}'
os.environ['COMFY_DIR'] = str(COMFY_DIR)
os.environ['FLOW_REGISTRY_PATH'] = str(FLOW_REGISTRY_PATH)
os.environ['LTX25_COMFY_LOG'] = str(COMFY_LOG)
os.environ['LTX25_COMFY_PID'] = str(ROOT / 'comfyui_ltx25.pid')
os.environ['LTX25_OUTPUT_DIR'] = str(LTX_OUTPUT_DIR)

'''
Validate user-editable local ports and remote queue limits.
確認教師設定的本機 Port 與遠端佇列限制有效，避免服務啟動後才失敗。
'''
assert 1024 <= REMOTE_PORT <= 65535, 'REMOTE_PORT 必須介於 1024～65535。'
assert 1 <= REMOTE_QUEUE_CAPACITY <= 1024, 'FIFO 佇列容量設定無效。'
assert 1 <= REMOTE_UPLOAD_CONCURRENCY <= 8, '上傳並發限制設定無效。'
assert REMOTE_RESULT_TTL_HOURS > 0 and REMOTE_RESULT_LIMIT_GIB > 0 and REMOTE_MIN_FREE_GIB >= 0
assert REMOTE_STALL_HOURS > 0
assert 1024 <= COMFY_PORT <= 65535 and COMFY_PORT != REMOTE_PORT
print('✓ 第 1 節完成：ComfyUI 持續執行；每支影片完成後自動清理模型與節點快取。')

assert 1 <= GPU_COUNT <= 4, 'GPU_COUNT 必須介於 1～4。'
assert all(0 <= gpu < GPU_COUNT for gpu in (MODEL_GPU_ID, AUX_GPU_ID)), 'GPU ID 超出 GPU_COUNT 的範圍。'
assert GPU_COUNT != 1 or MODEL_GPU_ID == AUX_GPU_ID, '單張 GPU 時，兩種元件應使用相同的 GPU ID。'


2. 安裝套件


In [ ]:
'''
Install ComfyUI and the custom nodes used by the LTX-2.5 workflow.
安裝 LTX-2.5 Workflow 會用到的 ComfyUI 與自訂節點。
'''
import os
import subprocess
import sys
from pathlib import Path
import torch

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

repos = [
    ('https://github.com/Comfy-Org/ComfyUI.git', COMFY_DIR, COMFY_REF),
    ('https://github.com/city96/ComfyUI-GGUF.git', COMFY_DIR / 'custom_nodes/ComfyUI-GGUF', None),
    ('https://github.com/Lightricks/ComfyUI-LTXVideo.git', COMFY_DIR / 'custom_nodes/ComfyUI-LTXVideo', None),
    ('https://github.com/kijai/ComfyUI-KJNodes.git', COMFY_DIR / 'custom_nodes/ComfyUI-KJNodes', None),
    ('https://github.com/yolain/ComfyUI-Easy-Use.git', COMFY_DIR / 'custom_nodes/ComfyUI-Easy-Use', None),
    ('https://github.com/SethRobinson/comfyui-workflow-to-api-converter-endpoint.git', COMFY_DIR / 'custom_nodes/workflow-to-api-converter', None),
]
UPDATE_CUSTOM_NODES = True
original_revisions = {}

'''
Install missing repositories and safely update existing ones.
安裝缺少的倉庫，並安全更新已存在的倉庫。
'''
for url, directory, ref in repos:
    if directory.exists() and not (directory / '.git').exists():
        raise RuntimeError(f'已有非 Git 資料夾：{directory}')
    if (directory / '.git').exists():
        original_revisions[directory] = subprocess.check_output(['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True).strip()
    else:
        command = ['git', '-c', 'advice.detachedHead=false', 'clone', '--depth', '1', '-q']

        '''
        Select the stable release for ComfyUI itself.
        ComfyUI 核心固定於穩定發行版。
        '''
        if ref:
            command += ['--branch', ref]
        subprocess.run(command + [url, str(directory)], check=True)
    if directory != COMFY_DIR and UPDATE_CUSTOM_NODES:
        if not subprocess.check_output(['git', '-C', str(directory), 'status', '--porcelain'], text=True).strip():
            subprocess.run(['git', '-C', str(directory), 'pull', '--ff-only', '-q'], check=True)
        else:
            print(f'略過已有本機修改的外掛：{directory.name}')

subprocess.run(['git', '-C', str(COMFY_DIR), 'fetch', '--depth', '1', 'origin', 'tag', COMFY_REF], check=True)
subprocess.run(['git', '-C', str(COMFY_DIR), 'checkout', '--quiet', '--detach', COMFY_REF], check=True)

'''
Indicate whether an existing model server must be restarted.
標記已啟動的模型服務是否需要重新啟動。
'''
LTX_CODE_UPDATED = any(
    directory in original_revisions and original_revisions[directory] != subprocess.check_output(
        ['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True,
    ).strip()
    for _, directory, _ in repos
)
for _, directory, _ in repos:
    print(directory.name, subprocess.check_output(['git', '-C', str(directory), 'rev-parse', '--short', 'HEAD'], text=True).strip())

constraint = ROOT / 'ltx25_torch_constraint.txt'

'''
Save the interpreter-specific compatibility limits.
儲存目前 Python 環境適用的相容性限制。
'''
constraint.write_text(f'torch=={torch.__version__}\ntransformers>=4.50.3,<5.0\n', encoding='utf-8')

for _, directory, _ in repos:
    requirement = directory / 'requirements.txt'
    if requirement.is_file():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), '-r', str(requirement)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), 'huggingface_hub', 'hf_xet', 'requests', 'Pillow', 'ipywidgets'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'kornia==0.8.2'], check=True)

legacy = COMFY_DIR / 'custom_nodes/ComfyUI-MultiGPU'
if legacy.is_dir():
    legacy.rename(ROOT / 'ComfyUI-MultiGPU.disabled')

print('ComfyUI installed:', COMFY_DIR)
print('Torch:', torch.__version__)
print('✓ 第 2 節完成：LTX-2.5 環境與套件已安裝。')


3. 設定環境


In [ ]:
'''
Import the packages required by the notebook.
匯入 Notebook 所需套件。
'''
import importlib.util
import os
import shutil
import subprocess
import time
from pathlib import Path
import ipywidgets as widgets
import requests
import torch
from IPython.display import FileLink, Video, display
from PIL import Image
MODEL_STORE.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_XET_CACHE'] = str(HF_CACHE_DIR / 'xet')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

'''
Require Kaggle's two-GPU runtime before loading the workflow.
載入 Workflow 前，確認 Kaggle 已啟用雙 GPU。
'''
assert torch.cuda.is_available() and torch.cuda.device_count() >= GPU_COUNT, '可用 CUDA GPU 數量不足；請檢查 NVIDIA 驅動、PyTorch 與 GPU_COUNT。'
for index in sorted({MODEL_GPU_ID, AUX_GPU_ID}):
    print(f'GPU {index}: {torch.cuda.get_device_name(index)}')

print('✓ 第 3 節完成：雙 GPU 裝置與環境設定已檢查。')


4. 模型與 Workflow


In [ ]:
'''
Fetch only the model files used by this Q4_K_M workflow.
只下載 Q4_K_M 雙階段 Workflow 真正需要的模型。
'''
import json
import shutil
import wave
from huggingface_hub import hf_hub_download
from PIL import Image
FLOW_REGISTRY = {
    "schema_version": 25,
    "models": {
        "transformers": {"q4km": {"filename": "LTX-2.5-Distilled-Q4_K_M.gguf"}},
        "text_encoder": {"basename": "gemma4-12b-ltx25-w4a8-v2.safetensors"},
        "video_vae": {"filename": "vae/ltx-2.5-video-vae-conv-bf16.safetensors"},
        "audio_vae": {"filename": "vae/ltx-2.5-audio-vae-bf16.safetensors"},
        "spatial_upscaler": {"filename": "latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors"},
    },
    "workflows": {"t2v_i2v": {"local_name": "LTX25_T2V_I2V.json"}},
}
model_files = [
    ("realrebelai/LTX-2.5_GGUFs", "LTX-2.5-Distilled-Q4_K_M.gguf", "diffusion_models"),
    ("realrebelai/Rebels_w4a8s", "LTX/ENCODERS/gemma4-12b-ltx25-w4a8-v2.safetensors", "text_encoders"),
    ("comfyicu/LTX-2.5", "vae/ltx-2.5-video-vae-conv-bf16.safetensors", "vae"),
    ("comfyicu/LTX-2.5", "vae/ltx-2.5-audio-vae-bf16.safetensors", "vae"),
    ("comfyicu/LTX-2.5", "latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors", "latent_upscale_models"),
]

'''
Link scratch weights into the model folders expected by ComfyUI.
透過 symlink 將暫存權重連到 ComfyUI 預期的模型資料夾。
'''
for repo, filename, folder in model_files:
    target = COMFY_DIR / "models" / folder / Path(filename).name
    stored = MODEL_STORE / folder / filename
    target.parent.mkdir(parents=True, exist_ok=True)
    stored.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file() and not target.is_symlink() and target.stat().st_size > 1_000_000:
        if not stored.exists():
            shutil.move(str(target), str(stored))
        elif target.stat().st_size == stored.stat().st_size:
            target.unlink()
        else:
            print("Existing model retained:", target.name)
            continue
    if not stored.is_file() or stored.stat().st_size <= 1_000_000:
        stored = Path(hf_hub_download(repo_id=repo, filename=filename, local_dir=str(MODEL_STORE / folder)))
    if target.is_symlink() or target.is_file():
        target.unlink()
    target.symlink_to(stored)
    print("Model:", target.name, "->", stored)

gguf_path = COMFY_DIR / "models/diffusion_models/LTX-2.5-Distilled-Q4_K_M.gguf"
multigpu_dir = COMFY_DIR / "models/unet_gguf"
multigpu_dir.mkdir(parents=True, exist_ok=True)
multigpu_link = multigpu_dir / gguf_path.name
if multigpu_link.is_symlink() and not multigpu_link.exists():
    multigpu_link.unlink()

if not multigpu_link.exists():
    multigpu_link.symlink_to(gguf_path)

workflow_file = hf_hub_download(
    repo_id="realrebelai/LTX-2.5_GGUFs",
    filename="LTX 2.5 GGUF (WORKFLOW).json",
    revision="112436f97aaf99ce13ecb7b7eca7e2f6c128d3ec",
    local_dir=str(SCRATCH_DIR / ".hf_workflow_stage"),
)
ui = json.loads(Path(workflow_file).read_text(encoding="utf-8"))

'''
Substitute model filenames throughout the UI workflow, including subgraphs.
在完整 UI Workflow（含子圖）中替換模型檔名。
'''
replacements = {
    "gguf": gguf_path.name,
    "text_encoder": FLOW_REGISTRY["models"]["text_encoder"]["basename"],
    "video_vae": Path(FLOW_REGISTRY["models"]["video_vae"]["filename"]).name,
    "audio_vae": Path(FLOW_REGISTRY["models"]["audio_vae"]["filename"]).name,
    "upscaler": Path(FLOW_REGISTRY["models"]["spatial_upscaler"]["filename"]).name,
}

def replace_names(value):
    '''
    Keep this transformation recursive because the workflow contains subgraphs.
    Workflow 包含巢狀子圖，因此需遞迴處理名稱。
    '''
    if isinstance(value, dict):
        return {key: replace_names(item) for key, item in value.items()}
    if isinstance(value, list):
        return [replace_names(item) for item in value]
    if not isinstance(value, str):
        return value
    lower = value.lower()
    if lower.endswith(".gguf") and "ltx" in lower:
        return replacements["gguf"]
    if "gemma4" in lower and lower.endswith(".safetensors") and "enhancer" not in lower:
        return replacements["text_encoder"]
    if "video-vae" in lower and lower.endswith(".safetensors"):
        return replacements["video_vae"]
    if "audio-vae" in lower and lower.endswith(".safetensors"):
        return replacements["audio_vae"]
    if "upscaler" in lower and lower.endswith(".safetensors"):
        return replacements["upscaler"]
    return value

INPUT_DIR = COMFY_DIR / "input"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
Image.new("RGB", (64, 64)).save(INPUT_DIR / "__ltx25_placeholder__.png")
with wave.open(str(INPUT_DIR / "__ltx25_placeholder__.wav"), "wb") as output_audio:
    output_audio.setnchannels(1)
    output_audio.setsampwidth(2)
    output_audio.setframerate(16000)
    output_audio.writeframes(b"\x00\x00" * 16000)

'''
Replace the sample filenames in nested LoadImage and LoadAudio nodes.
將子圖內的範例圖片與音訊名稱改成實際存在的預設檔。
'''
def patch_placeholders(graph):
    for node in graph.get("nodes", []):
        values = node.get("widgets_values")
        if node.get("type") == "LoadImage" and isinstance(values, list) and values:
            values[0] = "__ltx25_placeholder__.png"
        if node.get("type") == "LoadAudio" and isinstance(values, list) and values:
            values[0] = "__ltx25_placeholder__.wav"
    for child in graph.get("definitions", {}).get("subgraphs", []):
        patch_placeholders(child)

ui = replace_names(ui)
patch_placeholders(ui)
WORKFLOW_DIR = COMFY_DIR / "user/default/workflows/LTX25-Kaggle"
WORKFLOW_DIR.mkdir(parents=True, exist_ok=True)
(WORKFLOW_DIR / "LTX25_T2V_I2V.json").write_text(json.dumps(ui, ensure_ascii=False), encoding="utf-8")
FLOW_REGISTRY_PATH.write_text(json.dumps(FLOW_REGISTRY, ensure_ascii=False), encoding="utf-8")
print("Models and workflow ready.")
FLF_WORKFLOW_PATH = WORKFLOW_DIR / 'LTX25_FLF2V.json'
if not FLF_WORKFLOW_PATH.is_file():
    flf_url = 'https://raw.githubusercontent.com/UCHIJ/LTX2.5-first_to_last_frame/main/LTX%202.5-first_frame_to_last_frame.json'
    flf_response = requests.get(flf_url, timeout=90)
    flf_response.raise_for_status()
    flf_ui = flf_response.json()
    flf_nodes = list(flf_ui.get('nodes', []))
    for subgraph in flf_ui.get('definitions', {}).get('subgraphs', []):
        flf_nodes.extend(subgraph.get('nodes', []))
    flf_images = [node for node in flf_nodes if node.get('type') == 'LoadImage']
    flf_guides = [node for node in flf_nodes if node.get('type') == 'LTXVAddGuide']
    if len(flf_images) < 2 or len(flf_guides) < 4:
        raise RuntimeError('FLF2V Workflow does not contain two images and two-stage guides.')
    '''
    Normalize legacy FLF2V model names before the workflow is saved.
    儲存 FLF2V Workflow 前，將舊版模型檔名對應至已安裝的模型。
    '''
    flf_ui = replace_names(flf_ui)
    patch_placeholders(flf_ui)
    FLF_WORKFLOW_PATH.write_text(json.dumps(flf_ui, ensure_ascii=False), encoding='utf-8')

'''
Upgrade an existing downloaded FLF2V workflow without downloading it again.
修正先前已下載的 FLF2V Workflow，無須重新下載模型或 Workflow。
'''
flf_existing = json.loads(FLF_WORKFLOW_PATH.read_text(encoding='utf-8'))

'''
Keep all installed Video VAE references consistent, including nested subgraphs.
統一 Video VAE 檔名，包括巢狀子圖裡的設定。
'''
flf_corrected = replace_names(flf_existing)

'''
Persist the normalized workflow only when changes are necessary.
只有實際需要修正時才更新 Workflow 檔案。
'''
if flf_corrected != flf_existing:
    FLF_WORKFLOW_PATH.write_text(json.dumps(flf_corrected, ensure_ascii=False), encoding='utf-8')
    print('FLF2V 已修正舊版模型檔名。')

print('FLF2V workflow ready:', FLF_WORKFLOW_PATH.name)

'''
Install the original workflow conversion and output-handling engine locally.
在 Kaggle 本機建立原版 Workflow 轉換與輸出引擎。
'''
ENGINE_CODE = '\nimport asyncio\nimport io\nimport math\nfrom fractions import Fraction\nimport re\nimport wave\nimport json\nimport os\nimport shutil\nimport subprocess\nimport tempfile\nimport threading\nimport time\nimport uuid\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport requests\nfrom PIL import Image\n\nCOMFY_BASE = os.environ.get("COMFY_BASE", "http://127.0.0.1:8188").rstrip("/")\nMODEL_GPU_DEVICE = os.environ.get(\'LTX25_MODEL_GPU_DEVICE\', \'cuda:0\')\nAUX_GPU_DEVICE = os.environ.get(\'LTX25_AUX_GPU_DEVICE\', \'cuda:1\')\nCOMFY_DIR = Path(os.environ.get("COMFY_DIR", str(Path.home() / \'ai_course/ComfyUI\')))\nFLOW_REGISTRY_PATH = Path(os.environ.get("FLOW_REGISTRY_PATH", str(Path.home() / \'ai_course/ltx25_local_flows.json\')))\nWORKFLOW_DIR = Path(os.environ.get(\n    "WORKFLOW_DIR",\n    str(COMFY_DIR / "user/default/workflows/LTX25-Kaggle"),\n))\n\nif not FLOW_REGISTRY_PATH.is_file():\n    raise RuntimeError(f"Flow registry missing: {FLOW_REGISTRY_PATH}")\n\nFLOW_REGISTRY = json.loads(FLOW_REGISTRY_PATH.read_text(encoding="utf-8"))\n\nSTACK_SCHEMA_VERSION = int(FLOW_REGISTRY.get("schema_version", 0))\n\nMODEL_FILES = {\n    tier: item["filename"]\n    for tier, item in FLOW_REGISTRY["models"]["transformers"].items()\n}\nPRIMARY_MODEL_TIER = os.environ.get("LTX25_PRIMARY_MODEL_TIER", "q4km").lower()\nFALLBACK_MODEL_TIERS = [\n    x.strip().lower()\n    for x in os.environ.get("LTX25_FALLBACK_MODEL_TIERS", "").split(",")\n    if x.strip()\n]\n\nVAE_TILE_SIZE = int(os.environ.get("LTX25_VAE_TILE_SIZE", "512"))\nVAE_TILE_OVERLAP = int(os.environ.get("LTX25_VAE_TILE_OVERLAP", "64"))\nVAE_TEMPORAL_SIZE = int(os.environ.get("LTX25_VAE_TEMPORAL_SIZE", "32"))\nVAE_TEMPORAL_OVERLAP = int(os.environ.get("LTX25_VAE_TEMPORAL_OVERLAP", "8"))\n\nCOMFY_LOG_PATH = Path(os.environ.get("LTX25_COMFY_LOG", str(Path.home() / \'ai_course/comfyui_ltx25.log\')))\nCOMFY_PID_PATH = Path(os.environ.get("LTX25_COMFY_PID", str(Path.home() / \'ai_course/comfyui_ltx25.pid\')))\nTEXT_ENCODER = Path(FLOW_REGISTRY["models"]["text_encoder"]["basename"]).name\nVIDEO_VAE = Path(FLOW_REGISTRY["models"]["video_vae"]["filename"]).name\nAUDIO_VAE = Path(FLOW_REGISTRY["models"]["audio_vae"]["filename"]).name\nUPSCALER = Path(FLOW_REGISTRY["models"]["spatial_upscaler"]["filename"]).name\n\nT2V_I2V_WORKFLOW = WORKFLOW_DIR / FLOW_REGISTRY["workflows"]["t2v_i2v"]["local_name"]\nFLF2V_WORKFLOW = WORKFLOW_DIR / "LTX25_FLF2V.json"\n\nIMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}\nAUDIO_EXTENSIONS = {".wav", ".mp3", ".m4a", ".ogg"}\nMAX_IMAGE_BYTES = 15 * 1024 * 1024\nMAX_AUDIO_BYTES = 32 * 1024 * 1024\n\nJOBS = {}\nJOBS_LOCK = threading.Lock()\nEXECUTOR = ThreadPoolExecutor(max_workers=1)\n\n\n\ndef job_update(job_key, **fields):\n    # job_key is the dictionary key; fields may safely contain "job_id".\n    with JOBS_LOCK:\n        JOBS.setdefault(job_key, {}).update(fields)\n\n\ndef _comfy_http_error(method, path, response):\n    try:\n        body = response.json()\n        detail = json.dumps(body, ensure_ascii=False, indent=2)\n    except Exception:\n        detail = response.text[:12000]\n    raise RuntimeError(\n        f"ComfyUI {method} {path} failed: HTTP {response.status_code}\\n{detail}"\n    )\n\n\ndef comfy_get(path, **kwargs):\n    response = requests.get(f"{COMFY_BASE}/{path.lstrip(\'/\')}", **kwargs)\n    if not response.ok:\n        _comfy_http_error("GET", path, response)\n    return response\n\n\ndef comfy_post(path, **kwargs):\n    response = requests.post(f"{COMFY_BASE}/{path.lstrip(\'/\')}", **kwargs)\n    if not response.ok:\n        _comfy_http_error("POST", path, response)\n    return response\n\n\ndef get_gguf_loader_choices():\n    # Read only the GPU-selectable GGUF loader exposed by this notebook.\n    # 只檢查本 Notebook 註冊的雙 GPU GGUF Loader。\n    name = \'LTX25GGUFLoaderGPU\'\n    try:\n        spec = comfy_get(f\'/object_info/{name}\', timeout=30).json()[name]\n        return True, spec[\'input\'][\'required\'][\'unet_name\'][0], spec\n    except Exception as exc:\n        return False, [], {\'error\': str(exc)}\n\n\nRELEASE_AFTER_GENERATION = os.environ.get(\'LTX25_RELEASE_AFTER_GENERATION\', \'1\') == \'1\'\nSTOP_AFTER_GENERATION = os.environ.get(\'LTX25_STOP_AFTER_GENERATION\', \'1\') == \'1\'\nCOMFY_PID = int(os.environ[\'LTX25_COMFY_RUNTIME_PID\']) if os.environ.get(\'LTX25_COMFY_RUNTIME_PID\', \'\').isdigit() else None\n\n\ndef free_comfy():\n    """Request one cleanup only after the ComfyUI queue becomes idle."""\n\n    \'\'\'\n    Wait for the current workflow to leave the active queue before unloading.\n    等待目前影片工作完全離開執行佇列後，才允許卸載模型。\n    \'\'\'\n    try:\n        for _ in range(15):\n            queue = comfy_get(\'/queue\', timeout=5).json()\n            if not queue.get(\'queue_running\') and not queue.get(\'queue_pending\'):\n                break\n            time.sleep(1)\n        else:\n            print(\'警告：ComfyUI 仍有工作，為避免中斷其他影片，本次略過清理。\', flush=True)\n            return False\n\n        \'\'\'\n        Capture private ComfyUI RAM when the process ID is available.\n        若能取得 ComfyUI PID，就測量清理前後的專屬 RAM。\n        \'\'\'\n        process = None\n        before = None\n        if COMFY_PID is not None:\n            try:\n                import psutil\n                process = psutil.Process(COMFY_PID)\n                before = process.memory_full_info().uss / 1024**3\n            except (ImportError, OSError, RuntimeError) as exc:\n                print(f\'USS 診斷無法使用，仍會執行清理：{exc}\', flush=True)\n                process = None\n\n        \'\'\'\n        Unload ComfyUI models and clear node-result caches once.\n        只送出一次 ComfyUI 模型卸載與節點快取清理請求。\n        \'\'\'\n        comfy_post(\'/free\', json={\'unload_models\': True, \'free_memory\': True}, timeout=15)\n\n        \'\'\'\n        Allow asynchronous cleanup to finish before reporting memory usage.\n        等待背景清理完成，再回報記憶體使用狀態。\n        \'\'\'\n        time.sleep(5)\n        if process is not None:\n            after = process.memory_full_info().uss / 1024**3\n            print(f\'ComfyUI 清理請求已送出：USS {before:.2f} → {after:.2f} GiB；\'\n                  f\'變化 {before - after:+.2f} GiB。\', flush=True)\n        else:\n            print(\'ComfyUI 清理請求已送出（未取得 PID，無法量測 USS）。\', flush=True)\n        return True\n    except Exception as exc:\n        \'\'\'\n        Preserve a valid MP4 even when post-generation cleanup fails.\n        清理失敗時保留已生成的 MP4，並清楚顯示警告。\n        \'\'\'\n        print(f\'警告：ComfyUI 生成後清理失敗：{type(exc).__name__}: {exc}\', flush=True)\n        return False\n\n\ndef wait_comfy_ready(timeout=120):\n    deadline = time.time() + timeout\n    last = None\n    while time.time() < deadline:\n        try:\n            response = comfy_get("/system_stats", timeout=3)\n            if response.ok:\n                return True\n        except Exception as exc:\n            last = exc\n        time.sleep(1)\n    raise RuntimeError(f"ComfyUI is not ready: {last}")\n\n\ndef validate_dimensions(width, height):\n    """Reject nonintegral or unsupported LTX workflow input dimensions."""\n    if isinstance(width, bool) or isinstance(height, bool):\n        raise ValueError("Video dimensions must be integers")\n    try:\n        dimensions = (int(width), int(height))\n    except (TypeError, ValueError) as exc:\n        raise ValueError("Video dimensions must be integers") from exc\n    if any(str(value).strip() != str(number) for value, number in zip((width, height), dimensions)):\n        raise ValueError("Video dimensions must be integers without truncation")\n    if any(number < 256 or number > 4096 or number % 32 for number in dimensions):\n        raise ValueError("LTX width and height must be 256..4096 and divisible by 32")\n    return dimensions\n\n\ndef save_upload_bytes(data, suffix, prefix):\n    input_dir = COMFY_DIR / "input"\n    input_dir.mkdir(parents=True, exist_ok=True)\n    name = f"{prefix}_{uuid.uuid4().hex}{suffix.lower()}"\n    path = input_dir / name\n    path.write_bytes(data)\n    return name, path\n\n\ndef ensure_placeholder_image():\n    """\n    ComfyUI validates LoadImage nodes before execution even when the workflow\n    later disables image conditioning. Some downloaded UI workflows contain\n    stale example filenames such as Test__00139_.png.\n\n    Keep one valid tiny placeholder in ComfyUI/input and patch unused LoadImage\n    nodes to it. The workflow\'s use-image switch still controls conditioning.\n    """\n    input_dir = COMFY_DIR / "input"\n    input_dir.mkdir(parents=True, exist_ok=True)\n\n    name = "__ltx25_placeholder__.png"\n    path = input_dir / name\n\n    if not path.is_file():\n        Image.new("RGB", (64, 64), (0, 0, 0)).save(path, format="PNG")\n\n    return name\n\n\ndef ensure_placeholder_audio():\n    """Create a valid one-second silent WAV for unused LoadAudio validation."""\n    input_dir = COMFY_DIR / "input"\n    input_dir.mkdir(parents=True, exist_ok=True)\n    name = "__ltx25_placeholder__.wav"\n    path = input_dir / name\n    if not path.is_file():\n        with wave.open(str(path), "wb") as wf:\n            wf.setnchannels(1)\n            wf.setsampwidth(2)\n            wf.setframerate(16000)\n            wf.writeframes(b"\\x00\\x00" * 16000)\n    return name\n\n\ndef validate_image_bytes(data, suffix):\n    if not data:\n        raise ValueError("Image is empty.")\n    if len(data) > MAX_IMAGE_BYTES:\n        raise ValueError(f"Image exceeds {MAX_IMAGE_BYTES / 2**20:.0f} MiB limit.")\n    if suffix.lower() not in IMAGE_EXTENSIONS:\n        raise ValueError(f"Unsupported image extension: {suffix}")\n    try:\n        with Image.open(io.BytesIO(data)) as image:\n            image.verify()\n    except Exception as exc:\n        raise ValueError("Invalid/corrupted image.") from exc\n\n\ndef ffprobe_audio_bytes(data, suffix):\n    if not data:\n        raise ValueError("Audio is empty.")\n    if len(data) > MAX_AUDIO_BYTES:\n        raise ValueError(f"Audio exceeds {MAX_AUDIO_BYTES / 2**20:.0f} MiB limit.")\n    if suffix.lower() not in AUDIO_EXTENSIONS:\n        raise ValueError(f"Unsupported audio extension: {suffix}")\n\n    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:\n        tmp.write(data)\n        path = tmp.name\n\n    try:\n        result = subprocess.run(\n            [\n                "ffprobe", "-v", "error",\n                "-select_streams", "a:0",\n                "-show_entries", "stream=codec_name,profile:format=duration",\n                "-of", "json",\n                path,\n            ],\n            capture_output=True,\n            text=True,\n            timeout=30,\n        )\n        if result.returncode != 0:\n            raise ValueError("ffprobe could not decode the audio.")\n        payload = json.loads(result.stdout or "{}")\n        streams = payload.get("streams") or []\n        if not streams:\n            raise ValueError("No audio stream found.")\n        duration = float((payload.get("format") or {}).get("duration") or 0)\n        if duration <= 0:\n            raise ValueError("Could not determine audio duration.")\n        return {\n            "codec": streams[0].get("codec_name"),\n            "profile": streams[0].get("profile"),\n            "duration": duration,\n        }\n    finally:\n        try:\n            os.remove(path)\n        except OSError:\n            pass\n\n\ndef _norm_label(value):\n    text = str(value or "").strip().lower().replace("_", " ")\n    text = re.sub(r"[^a-z0-9]+", " ", text)\n    return " ".join(text.split())\n\n\nALIASES = {\n    "prompt": {"prompt", "positive prompt", "prompt positive", "text prompt", "prompt text", "main prompt", "positive text"},\n    "width": {"width", "video width", "output width"},\n    "height": {"height", "video height", "output height"},\n    "seconds": {"duration", "duration in seconds", "seconds", "video duration"},\n    "fps": {"fps", "frame rate", "frames per second"},\n    "seed": {"seed", "noise seed"},\n    "image_strength": {"img strength", "image strength"},\n    "video_vae": {"video vae"},\n    "audio_vae": {"audio vae"},\n    "text_encoder": {"text encoder", "encoder name", "clip name"},\n    "upscaler": {"upscale model", "upscaler", "spatial upscaler"},\n    "text_to_video": {"text to video", "t2v"},\n    "use_image": {"use image input", "use image", "image input"},\n    "use_audio": {"use custom audio", "custom audio", "use audio", "audio input"},\n    "prompt_enhance": {"enhance positive prompt", "prompt enhance", "enhance prompt"},\n    "tile_size": {"tile size"},\n    "tile_overlap": {"tile overlap"},\n    "temporal_size": {"temporal size"},\n    "temporal_overlap": {"temporal overlap"},\n    "model": {"ltx model", "diffusion model", "unet", "unet name", "model checkpoint"},\n}\nALIASES = {k: {_norm_label(x) for x in v} for k, v in ALIASES.items()}\n\n\ndef _matches_field_label(field, label):\n    label = _norm_label(label)\n    if label in ALIASES[field]:\n        return True\n    if field == "prompt" and "prompt" in label:\n        blocked = ("negative", "enhanc", "system", "instruction", "translated", "output")\n        return not any(word in label for word in blocked)\n    return False\n\n\ndef _iter_all_nodes(ui):\n    if not isinstance(ui, dict):\n        return\n    for node in ui.get("nodes", []):\n        if isinstance(node, dict):\n            yield ui, node\n    definitions = ui.get("definitions")\n    subgraphs = definitions.get("subgraphs", []) if isinstance(definitions, dict) else []\n    for subgraph in subgraphs:\n        if isinstance(subgraph, dict):\n            yield from _iter_all_nodes(subgraph)\n\n\ndef _remove_ui_link(owner, link_id):\n    links = owner.get("links")\n    if not isinstance(links, list) or link_id is None:\n        return\n    kept = []\n    for link in links:\n        if isinstance(link, list) and link and link[0] == link_id:\n            continue\n        if isinstance(link, dict) and link.get("id") == link_id:\n            continue\n        kept.append(link)\n    owner["links"] = kept\n\n\ndef _widget_inputs(node):\n    return [x for x in (node.get("inputs") or []) if isinstance(x, dict) and "widget" in x]\n\n\ndef _widget_index(node, input_name):\n    for i, inp in enumerate(_widget_inputs(node)):\n        if inp.get("name") == input_name:\n            return i\n    return None\n\n\ndef _coerce_widget_value(field, value, ui_type):\n    ui_type = str(ui_type or "").upper()\n    if ui_type == "BOOLEAN":\n        return bool(value)\n    if ui_type == "INT":\n        if field == "seconds":\n            return max(1, int(math.ceil(float(value))))\n        return int(round(float(value)))\n    if ui_type == "FLOAT":\n        return float(value)\n    if ui_type in {"STRING", "COMBO"}:\n        return str(value)\n    return value\n\n\ndef _set_exact_widget(owner, node, field, value, stats):\n    values = node.get("widgets_values")\n    if not isinstance(values, list):\n        return\n    for inp in node.get("inputs") or []:\n        if not isinstance(inp, dict):\n            continue\n        label = _norm_label(inp.get("label") or inp.get("name"))\n        if not _matches_field_label(field, label):\n            continue\n        idx = _widget_index(node, inp.get("name"))\n        if idx is None or idx >= len(values):\n            continue\n        # User-requested controls can be linked to helper nodes in a UI workflow\n        # (the current GGUF workflow links width/height to ResolutionSelector).\n        # Disconnect that incoming UI link so the patched widget value is the\n        # value that survives UI -> API conversion.\n        if field in {\n            "prompt", "width", "height", "seconds", "fps", "seed",\n            "image_strength", "text_to_video", "use_image", "use_audio",\n            "prompt_enhance", "tile_size", "tile_overlap",\n            "temporal_size", "temporal_overlap",\n        } and inp.get("link") is not None:\n            link_id = inp.get("link")\n            _remove_ui_link(owner, link_id)\n            inp["link"] = None\n            stats[f"disconnected_{field}"] = stats.get(f"disconnected_{field}", 0) + 1\n        values[idx] = _coerce_widget_value(field, value, inp.get("type"))\n        stats[field] = stats.get(field, 0) + 1\n\n\ndef _patch_primitives(node, values_by_field, stats):\n    values = node.get("widgets_values")\n    if not isinstance(values, list) or not values:\n        return\n    node_type = str(node.get("type") or "")\n    title = _norm_label(node.get("title"))\n    if node_type in {"PrimitiveString", "PrimitiveStringMultiline"} and _matches_field_label("prompt", title):\n        values[0] = values_by_field["prompt"]\n        stats["prompt"] = stats.get("prompt", 0) + 1\n    elif node_type in {"PrimitiveFloat", "PrimitiveInt"} and title in ALIASES["seconds"]:\n        values[0] = int(math.ceil(values_by_field["seconds"])) if node_type == "PrimitiveInt" else float(values_by_field["seconds"])\n        stats["seconds"] = stats.get("seconds", 0) + 1\n    elif node_type in {"PrimitiveFloat", "PrimitiveInt"} and title in ALIASES["fps"]:\n        values[0] = int(values_by_field["fps"]) if node_type == "PrimitiveInt" else float(values_by_field["fps"])\n        stats["fps"] = stats.get("fps", 0) + 1\n    elif node_type == "PrimitiveInt" and title in ALIASES["seed"]:\n        values[0] = int(values_by_field["seed"])\n        stats["seed"] = stats.get("seed", 0) + 1\n\n\ndef patch_workflow(\n    ui, *, mode, prompt, image_name, last_image_name=None, audio_name, model_file,\n    width, height, seconds, fps, seed, image_strength,\n    tile_size, tile_overlap, temporal_size, temporal_overlap,\n):\n    text_mode = mode in {"t2v", "a2v"}\n    use_image = mode in {"i2v", "ia2v", "flf2v"}\n    use_audio = mode in {"a2v", "ia2v"}\n    values_by_field = {\n        "prompt": prompt, "width": width, "height": height, "seconds": seconds,\n        "fps": fps, "seed": seed, "image_strength": image_strength,\n        "video_vae": VIDEO_VAE, "audio_vae": AUDIO_VAE,\n        "text_encoder": TEXT_ENCODER, "upscaler": UPSCALER,\n        "text_to_video": text_mode, "use_image": use_image, "use_audio": use_audio,\n        "prompt_enhance": False, "tile_size": tile_size, "tile_overlap": tile_overlap,\n        "temporal_size": temporal_size, "temporal_overlap": temporal_overlap,\n        "model": model_file,\n    }\n    stats = {}\n\n    for owner, node in _iter_all_nodes(ui):\n        node_type = str(node.get("type") or "")\n        widgets = node.get("widgets_values")\n        _patch_primitives(node, values_by_field, stats)\n\n        if node_type == "LoadImage" and isinstance(widgets, list) and widgets:\n            # FLF2V assigns its ending image by the titled LoadImage node.\n            # FLF2V 依照 LoadImage 標題指定結尾圖片。\n            if mode == "flf2v" and "last" in _norm_label(node.get("title")):\n                widgets[0] = last_image_name\n                stats["last_image"] = stats.get("last_image", 0) + 1\n            else:\n                widgets[0] = image_name\n                stats["first_image"] = stats.get("first_image", 0) + 1\n            stats["load_image"] = stats.get("load_image", 0) + 1\n        elif node_type == "LoadAudio" and isinstance(widgets, list) and widgets:\n            widgets[0] = audio_name\n            stats["load_audio"] = stats.get("load_audio", 0) + 1\n        elif node_type in {"UnetLoaderGGUF", "UNETLoader"} and isinstance(widgets, list) and widgets:\n            widgets[0] = model_file\n            stats["model"] = stats.get("model", 0) + 1\n        elif node_type == "LatentUpscaleModelLoader" and isinstance(widgets, list) and widgets:\n            widgets[0] = UPSCALER\n            stats["upscaler"] = stats.get("upscaler", 0) + 1\n\n        for field, value in values_by_field.items():\n            _set_exact_widget(owner, node, field, value, stats)\n\n    # Contract checks: fail here rather than silently generating with stale defaults.\n    required = ["prompt", "width", "height", "seconds", "fps", "seed", "model"]\n    if mode in {"i2v", "ia2v", "flf2v"}:\n        required.append("load_image")\n    if mode == "flf2v":\n        required += ["first_image", "last_image"]\n    missing = [name for name in required if stats.get(name, 0) == 0]\n    if missing:\n        raise RuntimeError(\n            f"Workflow patch contract failed for mode={mode}: missing controls/nodes {missing}. "\n            f"Patch stats={stats}"\n        )\n    return ui, stats\n\n\n_OBJECT_INFO_CACHE = {"at": 0.0, "data": None}\n\n\ndef get_object_info_all(max_age=60):\n    now = time.time()\n    if _OBJECT_INFO_CACHE["data"] is None or now - _OBJECT_INFO_CACHE["at"] > max_age:\n        _OBJECT_INFO_CACHE["data"] = comfy_get("/object_info", timeout=60).json()\n        _OBJECT_INFO_CACHE["at"] = now\n    return _OBJECT_INFO_CACHE["data"]\n\n\ndef convert_workflow(ui_workflow):\n    response = comfy_post("/workflow/convert", json=ui_workflow, timeout=120)\n    payload = response.json()\n    if not isinstance(payload, dict) or not any(\n        isinstance(v, dict) and "class_type" in v for v in payload.values()\n    ):\n        raise RuntimeError(f"Workflow converter returned unexpected payload: {payload}")\n    return payload\n\n\ndef normalize_nested_random_noise(api_workflow):\n    """Use local Gaussian noise that preserves nested video/audio structure."""\n    converted = 0\n    for node in api_workflow.values():\n        if not isinstance(node, dict):\n            continue\n        if node.get("class_type") == "RandomNoise":\n            node["class_type"] = "LTX25NestedRandomNoise"\n            converted += 1\n    if converted == 0:\n        raise RuntimeError("Converted workflow contains no RandomNoise nodes to normalize.")\n    return converted\n\n\ndef normalize_api_workflow_for_gguf(api_workflow, model_file):\n    # Place GGUF diffusion on GPU 0 and encoders/VAEs on GPU 1.\n    # 將 GGUF Diffusion 指定至 GPU 0，文字編碼器與 VAE 指定至 GPU 1。\n    counts = {\'transformer\': 0, \'text_encoder\': 0, \'vae\': 0, \'upscaler\': 0}\n    for node in api_workflow.values():\n        kind = node.get(\'class_type\')\n        inputs = node.get(\'inputs\', {})\n        if kind in {\'UNETLoader\', \'UnetLoaderGGUF\'}:\n            node[\'class_type\'] = \'LTX25GGUFLoaderGPU\'\n            node[\'inputs\'] = {\'unet_name\': model_file, \'device\': MODEL_GPU_DEVICE}\n            counts[\'transformer\'] += 1\n        elif kind == \'CLIPLoader\':\n            node[\'class_type\'] = \'LTX25CLIPLoaderGPU\'\n            node[\'inputs\'] = {**inputs, \'device\': AUX_GPU_DEVICE}\n            counts[\'text_encoder\'] += 1\n        elif kind == \'VAELoader\':\n            \'\'\'\n            Correct a legacy FLF2V Video VAE name after UI-to-API conversion.\n            Workflow 轉換成 API 格式後，再修正舊版 FLF2V Video VAE 名稱。\n            \'\'\'\n            vae_name = inputs.get(\'vae_name\')\n            if isinstance(vae_name, str) and vae_name.startswith(\'ltx-2.5-video-vae-\') and vae_name != VIDEO_VAE:\n                inputs = {**inputs, \'vae_name\': VIDEO_VAE}\n                print(f\'Video VAE corrected: {vae_name} -> {VIDEO_VAE}\', flush=True)\n            node[\'class_type\'] = \'LTX25VAELoaderGPU\'\n            node[\'inputs\'] = {**inputs, \'device\': AUX_GPU_DEVICE}\n            counts[\'vae\'] += 1\n        elif kind == \'LatentUpscaleModelLoader\':\n            node[\'class_type\'] = \'LTX25UpscalerLoaderGPU\'\n            node[\'inputs\'] = {**inputs, \'device\': AUX_GPU_DEVICE}\n            counts[\'upscaler\'] += 1\n    # Reject a workflow that accidentally bypasses either GPU or the two-stage upscaler.\n    # 若 Workflow 略過任一 GPU 或缺少雙階段 Upscaler，直接拒絕執行。\n    if not all(counts.values()):\n        raise RuntimeError(f\'Missing dual-GPU component loaders in converted workflow: {counts}\')\n    print(\'Independent GPU loader mapping:\', counts)\n    return sum(counts.values())\n\n\ndef inject_frozen_external_audio(api_workflow, audio_name, seconds):\n    """Inject existing audio into both stages of the distilled AV workflow.\n\n    Local-only path:\n      LoadAudio -> TrimAudioDuration -> VAEEncodeAudio\n      -> LTXVSetAudioRefTokens -> frozen_audio\n      -> every active LTXVConcatAVLatent audio input\n\n    The normal positive/negative conditioning graph is left unchanged; only the\n    node\'s frozen_audio output is used. For audio-driven modes, both samplers are\n    switched to plain Euler.\n    """\n    if not audio_name:\n        raise RuntimeError("External audio injection requires audio_name.")\n\n    audio_vae_loader = None\n    conditioning_node = None\n    for node_id, node in api_workflow.items():\n        if not isinstance(node, dict):\n            continue\n        if node.get("class_type") == "VAELoader" and (node.get("inputs") or {}).get("vae_name") == AUDIO_VAE:\n            audio_vae_loader = str(node_id)\n        if conditioning_node is None and node.get("class_type") == "LTXVConditioning":\n            conditioning_node = str(node_id)\n\n    if audio_vae_loader is None:\n        raise RuntimeError(f"Audio VAELoader not found for {AUDIO_VAE!r}.")\n    if conditioning_node is None:\n        raise RuntimeError("LTXVConditioning not found in converted workflow.")\n\n    def fresh_id(base):\n        candidate = base\n        n = 1\n        while candidate in api_workflow:\n            n += 1\n            candidate = f"{base}_{n}"\n        return candidate\n\n    load_id = fresh_id("ltx25_external_audio")\n    trim_id = fresh_id("ltx25_trim_audio")\n    encode_id = fresh_id("ltx25_encode_audio")\n    freeze_id = fresh_id("ltx25_freeze_audio")\n\n    api_workflow[load_id] = {\n        "class_type": "LoadAudio",\n        "inputs": {"audio": audio_name},\n        "_meta": {"title": "LTX25 External Audio"},\n    }\n    api_workflow[trim_id] = {\n        "class_type": "TrimAudioDuration",\n        "inputs": {\n            "audio": [load_id, 0],\n            "start_index": 0.0,\n            "duration": float(seconds),\n        },\n        "_meta": {"title": "LTX25 Trim External Audio"},\n    }\n    api_workflow[encode_id] = {\n        "class_type": "VAEEncodeAudio",\n        "inputs": {\n            "audio": [trim_id, 0],\n            "vae": [audio_vae_loader, 0],\n        },\n        "_meta": {"title": "LTX25 Encode External Audio"},\n    }\n    api_workflow[freeze_id] = {\n        "class_type": "LTXVSetAudioRefTokens",\n        "inputs": {\n            "positive": [conditioning_node, 0],\n            "negative": [conditioning_node, 1],\n            "audio_latent": [encode_id, 0],\n        },\n        "_meta": {"title": "LTX25 Freeze External Audio"},\n    }\n\n    concat_count = 0\n    sampler_count = 0\n    for node in api_workflow.values():\n        if not isinstance(node, dict):\n            continue\n        class_type = node.get("class_type")\n        inputs = node.get("inputs")\n        if not isinstance(inputs, dict):\n            continue\n        if class_type == "LTXVConcatAVLatent":\n            # Official distilled A2V keeps the same encoded audio frozen in both stages.\n            inputs["audio_latent"] = [freeze_id, 2]\n            concat_count += 1\n        elif class_type == "KSamplerSelect":\n            inputs["sampler_name"] = "euler"\n            sampler_count += 1\n\n    if concat_count < 2:\n        raise RuntimeError(\n            "Expected a two-stage AV workflow with at least two LTXVConcatAVLatent nodes; "\n            f"found {concat_count}. The pinned workflow may have changed."\n        )\n    if sampler_count < 2:\n        raise RuntimeError(\n            "Expected at least two KSamplerSelect nodes for the two-stage audio path; "\n            f"found {sampler_count}."\n        )\n\n    return {\n        "frozen_audio_concat_nodes": concat_count,\n        "euler_samplers": sampler_count,\n    }\n\n\ndef _is_connection(value):\n    return (\n        isinstance(value, list) and len(value) == 2\n        and isinstance(value[0], (str, int)) and isinstance(value[1], int)\n    )\n\n\ndef validate_api_workflow(api_workflow):\n    """Structural validation against ComfyUI /object_info without queueing GPU work.\\n\\n    Supports normal required inputs plus newer grouped/dynamic inputs\\n    (for example Autogrow values.a / values.b / ...).\\n    """\n    object_info = get_object_info_all()\n    errors = []\n    for node_id, node in api_workflow.items():\n        if not isinstance(node, dict) or "class_type" not in node:\n            continue\n        class_type = node["class_type"]\n        spec = object_info.get(class_type)\n        if not isinstance(spec, dict):\n            errors.append(f"node {node_id}: class_type {class_type!r} is not registered")\n            continue\n        inputs = node.get("inputs") or {}\n        groups = spec.get("input") or {}\n        required = groups.get("required") or {}\n        optional = groups.get("optional") or {}\n        for name in required:\n            if name in inputs:\n                continue\n\n            # Switch branches are lazy/conditional inputs. Only the branch\n            # selected by `switch` is requested by current ComfyUI.\n            if class_type in {"ComfySwitchNode", "ComfySoftSwitchNode"} and name in {"on_false", "on_true"}:\n                switch_value = inputs.get("switch")\n                if isinstance(switch_value, bool):\n                    selected = "on_true" if switch_value else "on_false"\n                    if name != selected:\n                        continue\n                    # If the statically selected branch is missing, fall through\n                    # and report it. A linked/dynamic switch is deferred.\n                else:\n                    continue\n\n            # ComfyUI\'s newer schema supports grouped/dynamic inputs such as\n            # io.Autogrow.Input("values", ...). The converted API workflow does\n            # not contain a literal "values" key; it contains children such as\n            # "values.a", "values.b", ... instead.\n            #\n            # Treat the required group as present when at least one child key\n            # exists. This is generic and is not hard-coded to ComfyMathExpression.\n            dynamic_prefix = f"{name}."\n            has_dynamic_child = any(\n                isinstance(input_name, str) and input_name.startswith(dynamic_prefix)\n                for input_name in inputs\n            )\n            if has_dynamic_child:\n                continue\n\n            errors.append(f"node {node_id} {class_type}: missing required input {name!r}")\n\n        definitions = dict(optional)\n        definitions.update(required)\n        for name, value in inputs.items():\n            # LoadImage/LoadAudio file lists are dynamic. /object_info may have\n            # been cached before a request uploaded its file, so do not use the\n            # cached combo list as our own file-existence validator. ComfyUI\'s\n            # node validation remains authoritative when /prompt is queued.\n            if (class_type == "LoadAudio" and name == "audio") or (class_type == "LoadImage" and name == "image"):\n                continue\n\n            definition = definitions.get(name)\n            if not isinstance(definition, (list, tuple)) or not definition:\n                continue\n            choices = definition[0]\n            if isinstance(choices, list) and not _is_connection(value):\n                if value not in choices:\n                    preview = choices[:20]\n                    errors.append(\n                        f"node {node_id} {class_type}.{name}: {value!r} not in selectable choices {preview!r}"\n                    )\n    if errors:\n        raise RuntimeError("ComfyUI structural preflight failed:\\n" + "\\n".join(errors[:50]))\n    return True\n\n\ndef queue_prompt(api_workflow):\n    response = comfy_post(\n        "/prompt",\n        json={"prompt": api_workflow, "client_id": str(uuid.uuid4())},\n        timeout=120,\n    )\n    payload = response.json()\n    if payload.get("node_errors"):\n        raise RuntimeError(\n            "ComfyUI rejected workflow:\\n"\n            + json.dumps(payload["node_errors"], ensure_ascii=False, indent=2)\n        )\n    prompt_id = payload.get("prompt_id")\n    if not prompt_id:\n        raise RuntimeError(f"ComfyUI did not return prompt_id: {payload}")\n    return prompt_id\n\n\ndef comfy_crash_diagnostics():\n    parts = []\n\n    pid = None\n    try:\n        if COMFY_PID_PATH.is_file():\n            pid = int(COMFY_PID_PATH.read_text(encoding="utf-8").strip())\n            try:\n                os.kill(pid, 0)\n                parts.append(f"ComfyUI PID {pid} still exists.")\n            except OSError:\n                parts.append(f"ComfyUI PID {pid} is no longer running.")\n    except Exception as exc:\n        parts.append(f"PID check failed: {exc}")\n\n    try:\n        meminfo = {}\n        for line in Path("/proc/meminfo").read_text().splitlines():\n            if ":" in line:\n                key, value = line.split(":", 1)\n                meminfo[key] = value.strip()\n        parts.append(\n            "Host memory after failure: "\n            f"MemAvailable={meminfo.get(\'MemAvailable\', \'?\')}, "\n            f"SwapFree={meminfo.get(\'SwapFree\', \'?\')}."\n        )\n    except Exception:\n        pass\n\n    try:\n        if COMFY_LOG_PATH.is_file():\n            tail = COMFY_LOG_PATH.read_text(errors="replace")[-7000:]\n            parts.append("ComfyUI log tail:\\n" + tail)\n    except Exception as exc:\n        parts.append(f"Could not read ComfyUI log: {exc}")\n\n    return "\\n".join(parts)\n\n\ndef _looks_like_connection_loss(exc):\n    text = str(exc).lower()\n    return (\n        "connection refused" in text\n        or "failed to establish a new connection" in text\n        or "max retries exceeded" in text\n    )\n\n\ndef wait_prompt(prompt_id):\n    while True:\n        try:\n            payload = comfy_get(\n                f"/history/{prompt_id}",\n                timeout=60,\n            ).json()\n        except Exception as exc:\n            if _looks_like_connection_loss(exc):\n                raise RuntimeError(\n                    "ComfyUI became unreachable during generation. "\n                    "The ComfyUI process probably exited or was killed by memory pressure.\\n"\n                    + comfy_crash_diagnostics()\n                ) from exc\n            raise\n\n        if prompt_id in payload:\n            history = payload[prompt_id]\n            status = history.get("status", {})\n            if status.get("status_str") == "error":\n                # Show only the failing node and traceback, never megabytes of latent tensors.\n                # 只輸出失敗節點與 traceback，避免將大型 latent tensor 列印到畫面。\n                details = [entry[1] for entry in status.get("messages", []) if entry[0] == "execution_error"]\n                if details:\n                    error = details[-1]\n                    raise RuntimeError(\n                        f"ComfyUI node {error.get(\'node_id\')} ({error.get(\'node_type\')}): "\n                        f"{error.get(\'exception_type\')}: {error.get(\'exception_message\')}\\n"\n                        + "".join(error.get(\'traceback\', [])[-12:])\n                    )\n                raise RuntimeError(f"ComfyUI failed: {status.get(\'messages\', [])[-3:]}")\n            return history\n        time.sleep(3)\n\n\ndef extract_output_item(history):\n    # Prefer video-like outputs. Some nodes use "videos", older wrappers use\n    # "gifs", and newer/custom nodes may expose generic "files".\n    video_exts = {".mp4", ".webm", ".mov", ".mkv", ".gif"}\n    fallback = None\n\n    for output in history.get("outputs", {}).values():\n        if not isinstance(output, dict):\n            continue\n        for key in ("videos", "gifs", "images", "files"):\n            items = output.get(key, [])\n            if not isinstance(items, list):\n                continue\n            for item in items:\n                if not isinstance(item, dict) or not item.get("filename"):\n                    continue\n                suffix = Path(item["filename"]).suffix.lower()\n                if key in {"videos", "gifs"} or suffix in video_exts:\n                    return item\n                if fallback is None:\n                    fallback = item\n\n    if fallback is not None:\n        return fallback\n    raise RuntimeError("Generation completed but no output file was found in ComfyUI history.")\n\n\ndef materialize_output(job_id, item):\n    output_dir = Path(os.environ.get("LTX25_OUTPUT_DIR", str(Path.home() / "ai_course/ltx25_api_outputs")))\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    filename = item.get("filename", f"{job_id}.mp4")\n    suffix = Path(filename).suffix or ".mp4"\n    target = output_dir / f"{job_id}{suffix}"\n\n    response = comfy_get(\n        "/view",\n        params=item,\n        timeout=600,\n    )\n    target.write_bytes(response.content)\n    return target\n\n\ndef mux_original_audio(job_id, video_path, audio_name):\n    """Mux the original uploaded audio back onto the generated video.\n\n    The latent audio drives generation, but the delivered MP4 uses the user\'s\n    original waveform rather than a VAE-decoded reconstruction.\n    """\n    audio_path = COMFY_DIR / "input" / audio_name\n    if not audio_path.is_file():\n        raise RuntimeError(f"Uploaded audio disappeared: {audio_path}")\n\n    output_dir = Path(os.environ.get("LTX25_OUTPUT_DIR", str(Path.home() / "ai_course/ltx25_api_outputs")))\n    target = output_dir / f"{job_id}_with_input_audio.mp4"\n    cmd = [\n        "ffmpeg", "-y", "-v", "error",\n        "-i", str(video_path),\n        "-i", str(audio_path),\n        "-map", "0:v:0",\n        "-map", "1:a:0",\n        "-c:v", "copy",\n        "-c:a", "aac",\n        "-b:a", "192k",\n        "-shortest",\n        str(target),\n    ]\n    result = subprocess.run(cmd, capture_output=True, text=True)\n    if result.returncode != 0:\n        raise RuntimeError(f"ffmpeg audio mux failed: {result.stderr[:4000]}")\n    return target\n\n\n\n\'\'\'\nRead real encoded video properties; a completed model job alone is not proof.\n讀取影片實際編碼資訊，模型回報完成並不等於成品通過驗證。\n\'\'\'\ndef probe_video_contract(path):\n    """Return the measured video stream and duration using FFprobe."""\n    result = subprocess.run(\n        ["ffprobe", "-v", "error", "-select_streams", "v:0",\n         "-show_entries", "stream=width,height,avg_frame_rate,duration:format=duration",\n         "-of", "json", str(path)],\n        check=True, capture_output=True, text=True, timeout=120,\n    )\n    data = json.loads(result.stdout)\n    streams = data.get("streams") or []\n    if not streams:\n        raise ValueError("Generated file has no video stream")\n    video = streams[0]\n    duration = float(video.get("duration") or data.get("format", {}).get("duration") or 0)\n    rate = float(Fraction(video.get("avg_frame_rate", "0/1")))\n    if duration <= 0 or rate <= 0:\n        raise ValueError("Cannot verify video duration or FPS")\n    return {"width": int(video["width"]), "height": int(video["height"]),\n            "fps": rate, "duration": duration}\n\n\n\'\'\'\nHonor final output dimensions without sending invalid half-size LTX inputs.\n尊重使用者的最終影片尺寸，不把不合法的半尺寸送入 LTX 模型。\n\'\'\'\ndef finalize_video_contract(job_id, path, params):\n    """Verify raw output, disclose x2 scaling, and optionally normalize."""\n    requested_width, requested_height = validate_dimensions(params["width"], params["height"])\n    fps, seconds = int(params["fps"]), float(params["seconds"])\n    policy = params.get("output_policy", "exact_final")\n    if policy not in ("exact_final", "native"):\n        raise ValueError("output_policy must be exact_final or native")\n    if fps < 1 or fps > 60 or seconds <= 0:\n        raise ValueError("Invalid requested FPS or duration")\n    raw = probe_video_contract(path)\n    dimensions = (raw["width"], raw["height"])\n    requested = (requested_width, requested_height)\n    if dimensions not in (requested, (2 * requested_width, 2 * requested_height)):\n        raise ValueError(f"Unexpected raw LTX output {dimensions}; requested {requested}; do not silently resize")\n    tolerance_seconds = max(0.1, 2.0 / fps)\n    if abs(raw["fps"] - fps) > 0.01:\n        raise ValueError(f"LTX output FPS {raw[\'fps\']} differs from requested {fps}")\n    if abs(raw["duration"] - seconds) > tolerance_seconds:\n        raise ValueError(f"LTX output duration {raw[\'duration\']:.3f}s differs from {seconds}s by more than {tolerance_seconds:.3f}s")\n    delivered_path = Path(path)\n    processed = False\n    if policy == "exact_final" and dimensions != requested:\n        delivered_path = Path(path).with_name(Path(path).stem + "_final.mp4")\n        staging = delivered_path.with_suffix(".part.mp4")\n        try:\n            subprocess.run(\n                ["ffmpeg", "-y", "-v", "error", "-i", str(path),\n                 "-vf", f"scale={requested_width}:{requested_height}:flags=lanczos",\n                 "-c:v", "libx264", "-preset", "fast", "-crf", "18",\n                 "-pix_fmt", "yuv420p", "-c:a", "copy", "-movflags", "+faststart",\n                 str(staging)],\n                check=True, capture_output=True, text=True, timeout=600,\n            )\n            staging.replace(delivered_path)\n        finally:\n            staging.unlink(missing_ok=True)\n        processed = True\n    final = probe_video_contract(delivered_path)\n    if policy == "exact_final" and (final["width"], final["height"]) != requested:\n        raise ValueError("Normalized final video does not match requested dimensions")\n    if abs(final["fps"] - fps) > 0.01 or abs(final["duration"] - seconds) > tolerance_seconds:\n        raise ValueError("Final video FPS/duration verification failed")\n    audit = {\n        "requested": {"width": requested_width, "height": requested_height, "fps": fps, "seconds": seconds},\n        "provider_input": {"width": requested_width, "height": requested_height},\n        "raw_output": raw,\n        "delivered_output": final,\n        "spatial_scale": {"width": raw["width"] / requested_width, "height": raw["height"] / requested_height},\n        "output_policy": policy,\n        "postprocessed": processed,\n        "duration_tolerance_seconds": tolerance_seconds,\n    }\n    job_update(job_id, media_contract=audit)\n    print("LTX media contract:", json.dumps(audit, ensure_ascii=False), flush=True)\n    return delivered_path\n\n\ndef looks_like_oom(exc):\n    text = str(exc).lower()\n    needles = [\n        "cuda out of memory",\n        "out of memory",\n        "not enough memory",\n        "failed to allocate",\n        "cublas_status_alloc_failed",\n        "cudaerror: out of memory",\n    ]\n    return any(n in text for n in needles)\n\n\ndef choose_workflow(mode):\n    if mode == "flf2v":\n        if not FLF2V_WORKFLOW.is_file():\n            raise RuntimeError(f"FLF2V workflow missing: {FLF2V_WORKFLOW}")\n        return FLF2V_WORKFLOW\n    if mode not in {"t2v", "i2v", "a2v", "ia2v"}:\n        raise RuntimeError(f"Unsupported mode: {mode}")\n    if not T2V_I2V_WORKFLOW.is_file():\n        raise RuntimeError(f"Base workflow missing: {T2V_I2V_WORKFLOW}")\n    return T2V_I2V_WORKFLOW\n\n\ndef prepare_api_workflow(params, tier):\n    mode = params["mode"]\n    model_file = MODEL_FILES[tier]\n    workflow_path = choose_workflow(mode)\n    validate_dimensions(params["width"], params["height"])\n    ui = json.loads(workflow_path.read_text(encoding="utf-8"))\n    ui, patch_stats = patch_workflow(\n        ui,\n        mode=mode,\n        prompt=params["prompt"],\n        image_name=params["image_name"],\n        last_image_name=params.get("last_image_name"),\n        audio_name=params["audio_name"],\n        model_file=model_file,\n        width=params["width"],\n        height=params["height"],\n        seconds=params["seconds"],\n        fps=params["fps"],\n        seed=params["seed"],\n        image_strength=params["image_strength"],\n        tile_size=params["vae_tile_size"],\n        tile_overlap=params["vae_tile_overlap"],\n        temporal_size=params["vae_temporal_size"],\n        temporal_overlap=params["vae_temporal_overlap"],\n    )\n    api_workflow = convert_workflow(ui)\n    nested_noise_nodes = normalize_nested_random_noise(api_workflow)\n    patch_stats["nested_random_noise"] = nested_noise_nodes\n    normalized_loaders = normalize_api_workflow_for_gguf(api_workflow, model_file)\n    if mode == "flf2v":\n        # Both stages must condition on the opening and closing frames.\n        # 兩個階段都必須加入起始與結尾影格條件。\n        guides = [node for node in api_workflow.values() if node.get("class_type") == "LTXVAddGuide"]\n        frame_ids = [node.get("inputs", {}).get("frame_idx") for node in guides]\n        if frame_ids.count(0) < 2 or frame_ids.count(-1) < 2:\n            raise RuntimeError(f"FLF2V requires first/last guides in both stages: {frame_ids}")\n        image_names = {node.get("inputs", {}).get("image") for node in api_workflow.values()\n                       if node.get("class_type") == "LoadImage"}\n        if not {params["image_name"], params["last_image_name"]}.issubset(image_names):\n            raise RuntimeError(f"FLF2V did not bind both images: {image_names}")\n    if normalized_loaders:\n        patch_stats["api_unet_to_gguf"] = normalized_loaders\n    if mode in {"a2v", "ia2v"}:\n        audio_stats = inject_frozen_external_audio(\n            api_workflow, params["audio_name"], params["seconds"]\n        )\n        patch_stats.update(audio_stats)\n    validate_api_workflow(api_workflow)\n    return workflow_path, api_workflow, patch_stats\n\n\n\n\ndef stop_comfy_process():\n    """Stop the current ComfyUI process after the queue becomes idle."""\n    if COMFY_PID is None:\n        print(\'未設定 ComfyUI PID，略過停止服務。\', flush=True)\n        return False\n    try:\n        import psutil\n        process = psutil.Process(COMFY_PID)\n    except Exception as exc:\n        print(f\'無法取得 ComfyUI 行程，略過停止服務：{exc}\', flush=True)\n        return False\n    try:\n        for _ in range(15):\n            queue = comfy_get(\'/queue\', timeout=5).json()\n            if not queue.get(\'queue_running\') and not queue.get(\'queue_pending\'):\n                break\n            time.sleep(1)\n        else:\n            print(\'警告：ComfyUI 仍有工作，略過停止服務。\', flush=True)\n            return False\n    except Exception as exc:\n        if process.is_running():\n            print(f\'警告：無法確認 ComfyUI 工作是否完成，為避免中斷輸出，本次不停止服務：{exc}\', flush=True)\n            return False\n        print(\'ComfyUI 已不在執行，無須再次停止。\', flush=True)\n        return True\n    try:\n        process.terminate()\n        process.wait(timeout=30)\n        print(\'ComfyUI 已停止，將於下一次生成前自動重新啟動。\', flush=True)\n        return True\n    except Exception as exc:\n        print(f\'停止 ComfyUI 失敗：{exc}\', flush=True)\n        return False\n\ndef run_once(job_id, params, tier):\n    model_file = MODEL_FILES[tier]\n\n    loader_ready, gguf_choices, diagnostic = get_gguf_loader_choices()\n    if not loader_ready:\n        message = diagnostic.get(\'error\', diagnostic)\n        raise RuntimeError(\n            \'ComfyUI 的 LTX25GGUFLoaderGPU 尚未就緒；請先重新啟動 ComfyUI 啟動 Cell。\'\n            f\' 診斷資訊：{message}\'\n        )\n    if model_file not in gguf_choices:\n        raise RuntimeError(\n            f"Model exists on disk but LTX25GGUFLoaderGPU cannot select {model_file!r}. "\n            f"Choices={gguf_choices}"\n        )\n\n    job_update(job_id, status="preparing", message=f"Patching/validating workflow with {tier}.")\n    workflow_path, api_workflow, patch_stats = prepare_api_workflow(params, tier)\n    job_update(\n        job_id,\n        workflow=str(workflow_path),\n        patch_stats=patch_stats,\n        message=f"Workflow structurally validated with {tier}; queueing ComfyUI.",\n    )\n\n    prompt_id = queue_prompt(api_workflow)\n    job_update(\n        job_id,\n        prompt_id=prompt_id,\n        status="running",\n        message=f"ComfyUI is generating with {tier}.",\n    )\n\n    \'\'\'\n    Finish video output before releasing the models; always clean up after a submitted job.\n    先完成影片輸出，工作結束後再清理模型；失敗時也會嘗試安全清理。\n    \'\'\'\n    try:\n        history = wait_prompt(prompt_id)\n        item = extract_output_item(history)\n        result_path = materialize_output(job_id, item)\n        if params["mode"] in {"a2v", "ia2v"}:\n            result_path = mux_original_audio(job_id, result_path, params["audio_name"])\n        return finalize_video_contract(job_id, result_path, params)\n    finally:\n        if RELEASE_AFTER_GENERATION:\n            free_comfy()\n        if STOP_AFTER_GENERATION:\n            stop_comfy_process()\n\n\ndef worker(job_id, params):\n    try:\n        job_update(job_id, status="preparing", message="Preparing local ComfyUI workflow.")\n        # Model policy comes from this Kaggle notebook.\n        configured_chain = [PRIMARY_MODEL_TIER] + [\n            tier for tier in FALLBACK_MODEL_TIERS\n            if tier != PRIMARY_MODEL_TIER\n        ]\n\n        for tier in configured_chain:\n            if tier not in MODEL_FILES:\n                raise RuntimeError(\n                    f"Notebook model policy contains unsupported tier {tier!r}. "\n                    f"Supported: {sorted(MODEL_FILES)}"\n                )\n\n        # Skip configured fallback tiers that have not been downloaded.\n        available_chain = []\n        for tier in configured_chain:\n            model_path = COMFY_DIR / "models/diffusion_models" / MODEL_FILES[tier]\n            if model_path.is_file():\n                available_chain.append(tier)\n\n        if not available_chain:\n            raise RuntimeError(\n                "No configured model tier is downloaded. "\n                f"Notebook policy: {configured_chain}"\n            )\n\n        result = None\n        last_error = None\n        for index, tier in enumerate(available_chain):\n            try:\n                result = run_once(job_id, params, tier)\n                break\n            except Exception as exc:\n                last_error = exc\n                is_last = index == len(available_chain) - 1\n                if is_last or not looks_like_oom(exc):\n                    raise\n\n                next_tier = available_chain[index + 1]\n                job_update(\n                    job_id,\n                    status="retrying",\n                    active_model_tier=next_tier,\n                    message=f"{tier} failed with an OOM-like error; retrying {next_tier}.",\n                    last_tier_error=str(exc),\n                )\n\n        if result is None:\n            raise RuntimeError(f"All model tiers failed. Last error: {last_error}")\n\n        job_update(\n            job_id,\n            status="completed",\n            result_path=str(result),\n            message="Generation completed.",\n        )\n\n    except Exception as exc:\n        job_update(\n            job_id,\n            status="failed",\n            error=str(exc),\n            message="Generation failed.",\n        )\n\n\ndef _sample_preflight_params(mode):\n    return {\n        "mode": mode,\n        "prompt": "LTX-2.5 structural preflight",\n        "width": 608,\n        "height": 352,\n        "seconds": 3.0,\n        "fps": 24,\n        "seed": 42,\n        "image_strength": 0.7,\n        "vae_tile_size": VAE_TILE_SIZE,\n        "vae_tile_overlap": VAE_TILE_OVERLAP,\n        "vae_temporal_size": VAE_TEMPORAL_SIZE,\n        "vae_temporal_overlap": VAE_TEMPORAL_OVERLAP,\n        "image_name": ensure_placeholder_image(),\n        "audio_name": ensure_placeholder_audio(),\n        "audio_info": None,\n    }\n\n\n_PREFLIGHT_CACHE = {"at": 0.0, "data": None}\n\n\ndef preflight_all_modes(max_age=30):\n    now = time.time()\n    if _PREFLIGHT_CACHE["data"] is not None and now - _PREFLIGHT_CACHE["at"] <= max_age:\n        return _PREFLIGHT_CACHE["data"]\n    result = {}\n    for mode in ("t2v", "i2v", "a2v", "ia2v"):\n        try:\n            workflow_path, _, stats = prepare_api_workflow(\n                _sample_preflight_params(mode), PRIMARY_MODEL_TIER\n            )\n            result[mode] = {\n                "ok": True,\n                "workflow": str(workflow_path),\n                "patch_stats": stats,\n            }\n        except Exception as exc:\n            result[mode] = {\n                "ok": False,\n                "workflow": str(T2V_I2V_WORKFLOW),\n                "error": str(exc),\n            }\n    _PREFLIGHT_CACHE["at"] = now\n    _PREFLIGHT_CACHE["data"] = result\n    return result\n\n\n'
ENGINE_PATH = ROOT / "ltx25_kaggle_engine.py"
ENGINE_PATH.write_text(ENGINE_CODE, encoding="utf-8")
print("Original workflow engine + dual-GPU loader patch ready.")
LTX25_NOISE_DIR = COMFY_DIR / "custom_nodes/LTX25LocalNoise"
LTX25_NOISE_DIR.mkdir(parents=True, exist_ok=True)
LTX25_NOISE_CODE = 'import numpy as np\nimport torch\nimport comfy.nested_tensor\n\n\ndef _noise_for_tensor(tensor, generator, noise_inds=None):\n    if noise_inds is None:\n        return torch.randn(\n            tensor.size(), dtype=torch.float32, layout=tensor.layout,\n            generator=generator, device="cpu"\n        ).to(dtype=tensor.dtype)\n\n    unique_inds, inverse = np.unique(noise_inds, return_inverse=True)\n    noises = []\n    for i in range(unique_inds[-1] + 1):\n        noise = torch.randn(\n            [1] + list(tensor.size())[1:], dtype=torch.float32,\n            layout=tensor.layout, generator=generator, device="cpu"\n        ).to(dtype=tensor.dtype)\n        if i in unique_inds:\n            noises.append(noise)\n    noises = [noises[i] for i in inverse]\n    return torch.cat(noises, axis=0)\n\n\nclass _LTX25NestedNoise:\n    def __init__(self, seed):\n        self.seed = int(seed)\n\n    def generate_noise(self, input_latent):\n        latent = input_latent["samples"]\n        noise_inds = input_latent.get("batch_index")\n        generator = torch.manual_seed(self.seed)\n\n        if getattr(latent, "is_nested", False):\n            parts = [\n                _noise_for_tensor(part, generator, noise_inds)\n                for part in latent.unbind()\n            ]\n            return comfy.nested_tensor.NestedTensor(parts)\n\n        return _noise_for_tensor(latent, generator, noise_inds)\n\n\nclass LTX25NestedRandomNoise:\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {\n            "required": {\n                "noise_seed": (\n                    "INT",\n                    {\n                        "default": 0,\n                        "min": 0,\n                        "max": 0xFFFFFFFFFFFFFFFF,\n                        "control_after_generate": True,\n                    },\n                )\n            }\n        }\n\n    RETURN_TYPES = ("NOISE",)\n    FUNCTION = "get_noise"\n    CATEGORY = "LTX25/local"\n\n    def get_noise(self, noise_seed):\n        return (_LTX25NestedNoise(noise_seed),)\n\n\nNODE_CLASS_MAPPINGS = {"LTX25NestedRandomNoise": LTX25NestedRandomNoise}\nNODE_DISPLAY_NAME_MAPPINGS = {\n    "LTX25NestedRandomNoise": "LTX25 Nested Random Noise"\n}\n'
(LTX25_NOISE_DIR / "__init__.py").write_text(LTX25_NOISE_CODE, encoding="utf-8")

'''
Install a small standalone device-placement node pack, not the archived package.
安裝獨立的 GPU 裝置選擇節點，不使用封存的套件。
'''
DUAL_NODE_DIR = COMFY_DIR / 'custom_nodes/zz_ltx25_gpu_components'
DUAL_NODE_DIR.mkdir(parents=True, exist_ok=True)
DUAL_NODE_SOURCE = '# Select independent GPUs without archived DisTorch2 or model sharding.\n# 不使用封存的 DisTorch2，以元件方式分配兩張 GPU。\nfrom contextlib import nullcontext\nfrom contextvars import ContextVar\n\n# Access ComfyUI\'s existing model-loading functions.\n# 使用 ComfyUI 原有的模型載入方法。\nimport comfy.model_management as mm\n# Access the original ComfyUI tensor-loading function.\n# 取得 ComfyUI 原本的張量載入函式。\nimport comfy.memory_management as memory_management\n\n# Preserve the original reader for every other model.\n# 其他模型仍沿用原本的讀取方式。\n_original_read_tensor = memory_management.read_tensor_file_slice_into\n\n# Fall back to PyTorch copying for the LTX spatial upscaler only.\n# 僅讓 LTX Spatial Upscaler 改用 PyTorch 複製，避開 AIMDO 傳輸錯誤。\ndef _ltx25_read_tensor(tensor, destination, stream=None, destination2=None):\n    # Locate the checkpoint associated with this tensor, if available.\n    # 取得此張量所屬的權重檔案路徑。\n    tensor_storage = tensor.untyped_storage() if hasattr(tensor, \'untyped_storage\') else None\n    # Check the source metadata attached to an mmap-backed tensor.\n    # 讀取 mmap 張量所附帶的來源資訊。\n    source_path = getattr(tensor_storage, \'_comfy_source_path\', None)\n    # Use the original file handle when source metadata is absent.\n    # 若沒有來源標記，改從原始檔案控制代碼判斷。\n    if source_path is None:\n        # Read the file-slice descriptor used by AIMDO.\n        # 讀取 AIMDO 使用的檔案區段資訊。\n        file_slice = getattr(tensor_storage, \'_comfy_tensor_file_slice\', None)\n        # Derive a filename only when file-slice metadata exists.\n        # 只有在存在檔案區段資訊時，才取得檔案名稱。\n        source_path = getattr(getattr(file_slice, \'file_ref\', None), \'name\', \'\')\n    # Redirect only the known failing LTX-2.5 upscaler weights.\n    # 只對先前出錯的 LTX-2.5 Upscaler 權重採用替代路徑。\n    if \'ltx-2.5-latent-spatial-upscaler\' in str(source_path).lower():\n        # Report the bypass only on its first actual use.\n        # 只在第一次確實使用替代載入路徑時顯示訊息。\n        if not getattr(_ltx25_read_tensor, \'_reported\', False):\n            # Leave one diagnostic line in the ComfyUI log.\n            # 在 ComfyUI Log 保留一行診斷訊息。\n            print(\'[LTX25] Spatial Upscaler: bypassing AIMDO file-to-device transfer\')\n            # Suppress repeated messages for later weight tensors.\n            # 後續張量不再重複印出訊息。\n            _ltx25_read_tensor._reported = True\n        # Let ComfyUI\'s native tensor copy handle the mmap-backed weight.\n        # 交由 ComfyUI 後續的 PyTorch 張量複製流程處理 mmap 權重。\n        return False\n    # Keep AIMDO\'s efficient path for all other models.\n    # 其他模型保留 AIMDO 的低 RAM 載入路徑。\n    return _original_read_tensor(tensor, destination, stream=stream, destination2=destination2)\n\n# Register the targeted fallback before any model is loaded.\n# 在載入任何模型之前，註冊這項局部替代機制。\nmemory_management.read_tensor_file_slice_into = _ltx25_read_tensor\n\nimport folder_paths\nimport torch\nfrom nodes import NODE_CLASS_MAPPINGS as CORE_NODES\n\n# Keep device selection local to a single loader execution.\n# 讓裝置選擇只影響目前執行的 Loader，不改變其他節點的預設行為。\n_active_device = ContextVar(\'ltx25_device\', default=None)\n_original_torch_device = mm.get_torch_device\n_original_text_device = mm.text_encoder_device\n_original_vae_device = mm.vae_device\n\n# Return the selected device only when a loader requests one explicitly.\n# 只有 Loader 明確指定裝置時，才覆寫原本的裝置查詢。\ndef _selected_device(original):\n    selected = _active_device.get()\n    return torch.device(selected) if selected is not None else original()\n\n# Patch the three official device lookup functions without changing memory accounting.\n# 僅覆寫 ComfyUI 三個裝置查詢函式，不改寫記憶體管理程式。\nmm.get_torch_device = lambda: _selected_device(_original_torch_device)\nmm.text_encoder_device = lambda: _selected_device(_original_text_device)\nmm.vae_device = lambda: _selected_device(_original_vae_device)\n\n# Declare the common GPU choices used by every component loader.\n# 統一定義各模型元件使用的 GPU 選項。\nDEVICES = [f\'cuda:{index}\' for index in range(torch.cuda.device_count())] + [\'cpu\']\n\n# Convert a new-style ComfyNode output into the legacy tuple format.\n# 將新版 ComfyNode 的 NodeOutput 轉成舊式節點可使用的 tuple。\ndef _as_tuple(result):\n    # Use the wrapped values directly when the node already returns a tuple.\n    # 若原本就回傳 tuple，直接使用，不額外轉換。\n    if isinstance(result, tuple):\n        return result\n    # Newer ComfyUI returns NodeOutput with its values stored in args.\n    # 新版 ComfyUI 的 NodeOutput 會將輸出值保存在 args。\n    if hasattr(result, \'args\'):\n        return tuple(result.args)\n    # Fail early rather than returning an invalid node payload.\n    # 如果輸出格式無法辨識，立即拋出錯誤，不讓壞資料流入 sampling。\n    raise TypeError(f\'Unsupported node output type: {type(result).__name__}\')\n\n# Locate an observable load_device field to verify the selected GPU.\n# 查找模型或 patcher 的 load_device，以確認實際裝置。\ndef _load_device(value):\n    # Check the public wrapper and common patcher locations.\n    # 檢查模型本體與常見的 patcher 欄位。\n    for obj in (value, getattr(value, \'patcher\', None), getattr(value, \'model_patcher\', None)):\n        # Skip containers that cannot report a model device.\n        # 忽略無法提供模型裝置的物件。\n        if obj is None:\n            continue\n        # Check explicit inference allocation before other placement fields.\n        # 優先確認模型實際指定的載入裝置。\n        found = getattr(obj, \'load_device\', None)\n        # Return the first explicit device from the patcher.\n        # 回傳第一個有效的載入裝置。\n        if found is not None:\n            return str(found)\n    # Report non-verifiable models rather than inventing a device.\n    # 若無法從物件確認裝置，不推測載入結果。\n    return None\n\n# Use either the legacy FUNCTION API or ComfyNode\'s execute method.\n# 同時支援舊式 FUNCTION 與新版 ComfyNode execute API。\ndef _load_component(name, device, **inputs):\n    # Find the actual core node instead of inheriting incompatible schemas.\n    # 直接取得原始節點，不繼承可能不相容的節點 Schema。\n    node = CORE_NODES.get(name)\n    # Reject missing plugins before sampling or GPU allocation.\n    # 若缺少必要外掛，先報錯，避免開始耗時生成。\n    if node is None:\n        raise RuntimeError(f\'Missing upstream ComfyUI node: {name}\')\n    # Set the target GPU only for this execution.\n    # 將 GPU 指定範圍限制在目前 Loader 的執行過程。\n    token = _active_device.set(device)\n    # Make kernels that use torch.cuda.current_device follow the same GPU.\n    # 讓依賴 torch.cuda.current_device 的 CUDA kernel 也指向正確 GPU。\n    context = torch.cuda.device(device) if device.startswith(\'cuda:\') else nullcontext()\n    try:\n        # Execute original loader code with the selected CUDA device active.\n        # 使用原本的 Loader 程式，同時設定對應 CUDA 裝置。\n        with context:\n            # Prefer the legacy function when the node exposes one.\n            # 如果節點提供舊式 FUNCTION，優先沿用原本載入方法。\n            if hasattr(node, \'FUNCTION\'):\n                # Invoke the original node without passing the custom GPU selector.\n                # 不將自訂 GPU 參數傳入原始 Loader，避免參數不相容。\n                result = getattr(node(), node.FUNCTION)(**inputs)\n            # Handle newer ComfyNode definitions that only provide execute.\n            # 若使用新版 ComfyNode API，改呼叫 execute。\n            elif hasattr(node, \'execute\'):\n                # Use the actual core ComfyNode class method.\n                # 呼叫核心節點提供的新版執行方法。\n                result = node.execute(**inputs)\n            else:\n                # Stop at an incompatible upstream API instead of guessing.\n                # 遇到無法辨識的新版 API 就直接拋出錯誤，避免錯誤分配 GPU。\n                raise RuntimeError(f\'Unsupported upstream loader API: {name}\')\n        # Normalize node output and verify its load device if observable.\n        # 整理節點輸出，並在可取得資訊時核對載入裝置。\n        values = _as_tuple(result)\n        # Reject empty output from any model loader.\n        # 拒絕沒有回傳模型的 Loader。\n        if not values:\n            raise RuntimeError(f\'{name} returned no model\')\n        # Compare actual model device with the requested assignment.\n        # 將模型的實際裝置與指定 GPU 比對。\n        actual = _load_device(values[0])\n        # Fail immediately if ComfyUI selected the wrong device.\n        # 若實際裝置與預期不同，立即中止並提示。\n        if actual is not None and actual != device:\n            raise RuntimeError(f\'{name}: requested {device}, actual load_device={actual}\')\n        # Distinguish observed placement from selection-only cases.\n        # 區分已驗證載入位置與僅完成指定裝置的情況。\n        print(f\'[LTX25 dualGPU] {name}: requested={device}, observed={actual or "not exposed"}\')\n        # Return the result in a format compatible with old-style custom nodes.\n        # 使用舊式自訂節點可接收的 tuple 格式回傳。\n        return values\n    finally:\n        # Restore original device behavior for subsequent graph nodes.\n        # 在 Loader 結束後恢復其他節點的預設裝置選擇。\n        _active_device.reset(token)\n\n# Use the core GGUF loader while explicitly selecting GPU 0.\n# 使用 ComfyUI-GGUF 原有 Loader，明確指定 GPU 0。\nclass LTX25GGUFLoaderGPU:\n    # Expose the same model choices that the installed GGUF plugin recognizes.\n    # 使用目前 GGUF 外掛提供的模型清單。\n    @classmethod\n    def INPUT_TYPES(cls):\n        # Declare exact API names required by the existing workflow.\n        # 定義原始 Workflow 需要的 GGUF 模型名稱欄位。\n        return {\'required\': {\'unet_name\': (folder_paths.get_filename_list(\'unet_gguf\'),), \'device\': (DEVICES,)}}\n    RETURN_TYPES = (\'MODEL\',)\n    FUNCTION = \'load\'\n    CATEGORY = \'LTX25/dual_gpu\'\n\n    # Load a quantized LTX transformer through the upstream GGUF node.\n    # 透過 GGUF 外掛載入量化 LTX Transformer。\n    def load(self, unet_name, device):\n        # Reuse the GGUF implementation without any DisTorch2 modifications.\n        # 完全沿用 GGUF Loader，不加入 DisTorch2 的權重修補邏輯。\n        return _load_component(\'UnetLoaderGGUF\', device, unet_name=unet_name)\n\n# Use the standard CLIP loader with a separate GPU allocation.\n# 將 Gemma 文字編碼器分配到另一張 GPU。\nclass LTX25CLIPLoaderGPU:\n    # Keep the original CLIPLoader model-type options.\n    # 保留 ComfyUI 原本的 CLIPLoader 類型選項。\n    @classmethod\n    def INPUT_TYPES(cls):\n        # Extend the original choice list with the two CUDA devices.\n        # 在原本的文字模型選單加入雙 GPU 裝置選擇。\n        return {\'required\': {\'clip_name\': (folder_paths.get_filename_list(\'text_encoders\'),), \'type\': (CORE_NODES[\'CLIPLoader\'].INPUT_TYPES()[\'required\'][\'type\'][0],), \'device\': (DEVICES,)}}\n    RETURN_TYPES = (\'CLIP\',)\n    FUNCTION = \'load\'\n    CATEGORY = \'LTX25/dual_gpu\'\n\n    # Load the selected Gemma encoder without passing a conflicting device keyword.\n    # 不將自訂 device 參數直接傳入原版 CLIPLoader。\n    def load(self, clip_name, type, device):\n        # Leave the core optional device parameter at its default.\n        # 使用原版 Loader 的預設 optional device 參數。\n        return _load_component(\'CLIPLoader\', device, clip_name=clip_name, type=type)\n\n# Use ComfyUI\'s VAE loader for both video and audio weights.\n# Video 與 Audio VAE 都沿用 ComfyUI 的原版模型載入邏輯。\nclass LTX25VAELoaderGPU:\n    # Use the native VAE choice list, including any supported approximations.\n    # 保留 ComfyUI 原本完整的 VAE 清單。\n    @classmethod\n    def INPUT_TYPES(cls):\n        # Add explicit GPU selection to the existing VAE filename input.\n        # 在 VAE 檔名欄位加入指定 GPU 選項。\n        return {\'required\': {\'vae_name\': CORE_NODES[\'VAELoader\'].INPUT_TYPES()[\'required\'][\'vae_name\'], \'device\': (DEVICES,)}}\n    RETURN_TYPES = (\'VAE\',)\n    FUNCTION = \'load\'\n    CATEGORY = \'LTX25/dual_gpu\'\n\n    # Load VAE weights with the core loader in the selected CUDA context.\n    # 在指定 CUDA 裝置中呼叫原版 VAE Loader。\n    def load(self, vae_name, device):\n        # Reuse the core VAE loader for the chosen model.\n        # 透過原版 Loader 讀取選定的 VAE 權重。\n        return _load_component(\'VAELoader\', device, vae_name=vae_name)\n\n# Adapt the new-style LatentUpscaleModelLoader without inheriting ComfyNode.\n# 不繼承新版 ComfyNode，避免 LatentUpscaleModelLoader 缺少 INPUT_TYPES。\nclass LTX25UpscalerLoaderGPU:\n    # Supply a legacy-compatible input schema based on public folder paths.\n    # 使用目前 ComfyUI 支援的模型資料夾建立舊式相容 Schema。\n    @classmethod\n    def INPUT_TYPES(cls):\n        # Allow selection of the official LTX-2.5 spatial upscaler.\n        # 提供原生 LTX-2.5 Spatial Upscaler 模型選項。\n        return {\'required\': {\'model_name\': (folder_paths.get_filename_list(\'latent_upscale_models\'),), \'device\': (DEVICES,)}}\n    RETURN_TYPES = (\'LATENT_UPSCALE_MODEL\',)\n    FUNCTION = \'load\'\n    CATEGORY = \'LTX25/dual_gpu\'\n\n    # Call either the new execute method or old FUNCTION method as available.\n    # 依安裝的 ComfyUI 版本呼叫 execute 或原本的 FUNCTION。\n    def load(self, model_name, device):\n        # Keep the original spatial upscaler model-loading semantics.\n        # 保留原始 LatentUpscaleModelLoader 的模型載入邏輯。\n        return _load_component(\'LatentUpscaleModelLoader\', device, model_name=model_name)\n\n# Register the four standalone model loaders.\n# 註冊四種不依賴 MultiGPU 擴充套件的模型 Loader。\nNODE_CLASS_MAPPINGS = {\n    \'LTX25GGUFLoaderGPU\': LTX25GGUFLoaderGPU,\n    \'LTX25CLIPLoaderGPU\': LTX25CLIPLoaderGPU,\n    \'LTX25VAELoaderGPU\': LTX25VAELoaderGPU,\n    \'LTX25UpscalerLoaderGPU\': LTX25UpscalerLoaderGPU,\n}\n\n# Expose readable names in the local ComfyUI server.\n# 在 ComfyUI 本機服務顯示易讀的節點名稱。\nNODE_DISPLAY_NAME_MAPPINGS = {name: name for name in NODE_CLASS_MAPPINGS}\n'
(DUAL_NODE_DIR / '__init__.py').write_text(DUAL_NODE_SOURCE, encoding='utf-8')
print('Standalone GPU loader + targeted upscaler fallback installed:', DUAL_NODE_DIR)
print('✓ 第 4 節完成：模型與雙 GPU Workflow 元件已準備完成。')


5. 啟動 ComfyUI


In [ ]:
'''
Check and start the persistent ComfyUI service.
檢查並啟動可重複使用的 ComfyUI 服務。
'''
import requests
import time
import signal

'''
Check that all required dual-GPU loaders are registered.
確認雙 GPU 模型載入節點都已註冊。
'''
def loaders_ready():
    '''
    Use the ComfyUI node registry as the health check.
    以 ComfyUI 節點清單確認服務可用。
    '''
    try:
        for name in ('LTX25GGUFLoaderGPU', 'LTX25CLIPLoaderGPU', 'LTX25VAELoaderGPU', 'LTX25UpscalerLoaderGPU'):
            response = requests.get(f'http://127.0.0.1:{COMFY_PORT}/object_info/{name}', timeout=3)
            if not response.ok or name not in response.json():
                return False
        return True

    except requests.RequestException:
        return False


def ltx_stop_comfy_if_running(reason='教師手動停止'):
    '''
    Stop the current ComfyUI process if it is still running.
    若目前 ComfyUI 仍在執行，則安全停止。
    '''
    process = globals().get('COMFY_PROC')
    if process is None or process.poll() is not None:
        return False
    try:
        process.terminate()
        process.wait(timeout=30)
        print(f'ComfyUI 已停止（{reason}）。')
        return True
    except Exception as exc:
        print(f'停止 ComfyUI 失敗：{exc}')
        return False


def ltx_start_comfy(force_restart=False):
    '''
    Start ComfyUI and wait for all required LTX dual-GPU loaders.
    啟動 ComfyUI，並等待 LTX 雙 GPU 載入節點全部就緒。
    '''
    global COMFY_PROC, COMFY_CONFIG_ID
    previous = globals().get('COMFY_PROC')
    if force_restart and previous is not None and previous.poll() is None:
        ltx_stop_comfy_if_running('重新啟動服務')
    previous = globals().get('COMFY_PROC')
    if previous is not None and previous.poll() is None and globals().get('COMFY_CONFIG_ID') == CONFIG_ID and loaders_ready():
        os.environ['LTX25_COMFY_RUNTIME_PID'] = str(previous.pid)
        print('ComfyUI 已在執行，可直接使用。')
        return previous
    if previous is not None and previous.poll() is None:
        ltx_stop_comfy_if_running('設定變更')
    if loaders_ready():
        raise RuntimeError(f'{COMFY_PORT} 已有其他 ComfyUI 服務；請先停止該行程，再執行此 Cell。')
    command = [sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', str(COMFY_PORT), '--default-device', str(MODEL_GPU_ID), *STARTUP_FLAGS]
    server_env = os.environ.copy()
    server_env['LTX25_RELEASE_AFTER_GENERATION'] = '1' if RELEASE_AFTER_GENERATION else '0'
    server_env['LTX25_STOP_AFTER_GENERATION'] = '1' if STOP_COMFY_AFTER_GENERATION else '0'
    server_env.pop('PYTORCH_CUDA_ALLOC_CONF', None)
    server_env['PYTORCH_ALLOC_CONF'] = 'backend:native'
    with COMFY_LOG.open('w', encoding='utf-8') as stream:
        COMFY_PROC = subprocess.Popen(command, cwd=COMFY_DIR, env=server_env, stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(180):
        if COMFY_PROC.poll() is not None:
            raise RuntimeError(COMFY_LOG.read_text(errors='replace')[-6000:])
        if loaders_ready():
            os.environ['LTX25_COMFY_RUNTIME_PID'] = str(COMFY_PROC.pid)
            COMFY_CONFIG_ID = CONFIG_ID
            print(f'ComfyUI 已啟動：cache-{NODE_CACHE_MODE} / reserve-vram {GPU_VRAM_RESERVE_GIB} GiB。')
            return COMFY_PROC
        time.sleep(2)
    raise TimeoutError(COMFY_LOG.read_text(errors='replace')[-6000:])


def ltx_ensure_comfy_ready(force_restart=False):
    '''
    Ensure ComfyUI is alive and its required custom nodes are registered.
    確保 ComfyUI 存活，且必要的自訂節點已註冊。
    '''
    process = globals().get('COMFY_PROC')
    if force_restart or process is None or process.poll() is not None or not loaders_ready():
        return ltx_start_comfy(force_restart=force_restart)
    os.environ['LTX25_COMFY_RUNTIME_PID'] = str(process.pid)
    return process

assert NODE_CACHE_MODE in ('classic', 'none')
STARTUP_FLAGS = [
    '--reserve-vram', str(GPU_VRAM_RESERVE_GIB),
    '--cache-classic' if NODE_CACHE_MODE == 'classic' else '--cache-none',
    '--fast-disk',
    '--disable-comfy-compiler', '--disable-pinned-memory',
    '--disable-async-offload', '--fp16-unet',
]

'''
Estimate available RAM using host and cgroup limits when present.
同時考量主機與容器記憶體限制，估算實際可用 RAM。
'''
def ltx_available_ram_gib():
    memory = {}
    for line in Path('/proc/meminfo').read_text().splitlines():
        if ':' in line:
            key, value = line.split(':', 1)
            memory[key] = int(value.strip().split()[0])
    available = memory['MemAvailable'] * 1024
    cgroup = Path('/sys/fs/cgroup')
    limit_file = cgroup / 'memory.max'
    current_file = cgroup / 'memory.current'
    if limit_file.is_file() and current_file.is_file():
        limit = limit_file.read_text().strip()
        if limit.isdecimal():
            used = int(current_file.read_text().strip())
            cache_file = cgroup / 'memory.stat'
            inactive_file = 0
            if cache_file.is_file():
                for line in cache_file.read_text().splitlines():
                    if line.startswith('inactive_file '):
                        inactive_file = int(line.split()[1])
                        break
            available = min(available, max(0, int(limit) - used + inactive_file))
    return available / 1024**3

CONFIG_ID = ('ltx25-nvidia-gpu-v11-classroom', NODE_CACHE_MODE, GPU_VRAM_RESERVE_GIB, MODEL_GPU_ID, AUX_GPU_ID)
ltx_ensure_comfy_ready(force_restart=globals().get('LTX_CODE_UPDATED', False))
print('雙 GPU 模型載入節點：正常')
print('✓ 第 5 節完成：ComfyUI 服務及雙 GPU 節點已就緒。')


6. 圖片上傳工具與路徑管理


In [ ]:
'''
Save local images for I2V and FLF2V while preserving their filenames.
儲存 I2V 與 FLF2V 的本機圖片，並保留原始檔名。
'''
def save_selected_image(change, folder):
    '''
    Ignore events without an uploaded file.
    沒有選取檔案時，維持原本的圖片設定。
    '''
    uploaded = change['new']
    if not uploaded:
        return None
    if isinstance(uploaded, dict):
        filename, item = next(iter(uploaded.items()))
    else:
        item = uploaded[0]
        filename = item['name']
    filename = Path(filename.replace('\\', '/')).name
    destination = ROOT / 'ltx25_images' / folder / filename
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(bytes(item['content']))
    print('已選取：', destination)
    return str(destination)

print('✓ 第 6 節完成：圖片上傳工具已就緒。')


7. T2V：文字生成影片


In [ ]:
'''
Choose duration and resolution for this video task.
設定這個 Task 的秒數與輸出尺寸；解析度或長度增加會提高 GPU／RAM 用量。
'''
SECONDS = 10
WIDTH, HEIGHT = 288, 512
SEED = 42
IMAGE_STRENGTH = 0.7  # 0.0～1.0

'''
Define the prompt specifically for text-to-video generation.
設定只供文字生成影片使用的提示詞。
'''
T2V_PROMPT = '''
電影感多視角鏡頭序列，總時長 10 秒。主角始終是同一位烏黑直長髮、氣質溫柔的韓國年輕女大學生，身穿米白色寬鬆針織開襟衫，肩背帆布托特包；人物臉部特徵、服裝與髮型保持一致。
場景是清晨的櫻花林蔭大道，兩側盛開粉紅櫻花，淡淡晨霧籠罩，柔和金色陽光穿過樹梢，花瓣隨微風緩緩飄落。
0至3秒（側身遠景）：她靜靜站在櫻花大道旁，微微側身眺望遠方，鏡頭平穩取景。
3至6秒（正面中景）：鏡頭流暢移動到她的正前方並緩緩推近，她轉向鏡頭，神情自然柔和。
6至8秒（臉部特寫）：鏡頭繼續推近，她輕輕抬眼直視鏡頭，露出甜美而溫暖的微笑。
8至10秒（微笑與眨眼）：維持正面特寫與眼神接觸，她在最後一秒俏皮地單眼眨眼（wink），隨即保持微笑直到最後一格；微風吹動髮絲。
35mm 底片美學、連貫自然運鏡、浪漫淺景深、柔和粉彩色調、自然皮膚紋理與流暢動作。
'''.strip()
T2V_RUN_NAME = 'clip01'
ACTIVE_JOB = {'mode': 't2v', 'prompt': T2V_PROMPT, 'run_name': T2V_RUN_NAME}

'''
Confirm which mode the generation cell will use.
確認生成 Cell 將使用的模式。
'''
print('目前模式：T2V | 輸出名稱：', T2V_RUN_NAME)
print('✓ T2V 模式設定完成：尚未生成影片。')


8. I2V：上傳一張圖片


In [ ]:
'''
Choose duration and resolution for this video task.
設定這個 Task 的秒數與輸出尺寸；解析度或長度增加會提高 GPU／RAM 用量。
'''
SECONDS = 10
WIDTH, HEIGHT = 288, 512
SEED = 42
IMAGE_STRENGTH = 0.7  # 0.0～1.0

'''
Define the prompt specifically for image-to-video generation.
設定只供單張圖片生成影片使用的提示詞。
'''
I2V_PROMPT = '''
The person in the photograph begins to walk naturally, smooth cinematic camera movement.
'''.strip()
I2V_IMAGE_PATH = globals().get('I2V_IMAGE_PATH')
ACTIVE_JOB = {'mode': 'i2v', 'prompt': I2V_PROMPT, 'first_path': I2V_IMAGE_PATH}

'''
Keep this mode's selection separate from later mode changes.
將本模式的上傳狀態與後續模式切換隔離。
'''
i2v_job = ACTIVE_JOB

'''
Update this I2V job when a local photo is selected.
選取本機照片後，更新此 I2V 工作設定。
'''
def on_i2v_upload(change):
    '''
    Update the reusable I2V image path.
    更新下次也可沿用的 I2V 圖片路徑。
    '''
    global I2V_IMAGE_PATH
    selected = save_selected_image(change, 'i2v')
    if selected:
        I2V_IMAGE_PATH = selected
        i2v_job['first_path'] = selected
        print('影片名稱：', Path(selected).stem + '.mp4')

i2v_upload = widgets.FileUpload(accept='.png,.jpg,.jpeg,.webp', multiple=False, description='選擇 I2V 照片')
i2v_upload.observe(on_i2v_upload, names='value')

'''
Display the control for selecting a local image.
顯示本機圖片選取按鈕。
'''
display(i2v_upload)
print('目前模式：I2V | 圖片：', I2V_IMAGE_PATH or '請上傳照片')
print('✓ I2V 模式設定完成：請確認圖片已上傳，再執行 Generate。')


9. FLF2V：上傳開頭與結尾圖片


In [ ]:
'''
Choose duration and resolution for this video task.
設定這個 Task 的秒數與輸出尺寸；解析度或長度增加會提高 GPU／RAM 用量。
'''
SECONDS = 10
WIDTH, HEIGHT = 288, 512
SEED = 42
IMAGE_STRENGTH = 0.7  # 0.0～1.0

'''
Define the prompt specifically for first/last-frame generation.
設定只供首尾影格影片生成使用的提示詞。
'''
FLF2V_PROMPT = '''
Smooth cinematic motion transitioning naturally between the opening image and ending image.
'''.strip()
FLF2V_FIRST_IMAGE_PATH = globals().get('FLF2V_FIRST_IMAGE_PATH')
FLF2V_LAST_IMAGE_PATH = globals().get('FLF2V_LAST_IMAGE_PATH')
ACTIVE_JOB = {
    'mode': 'flf2v',
    'prompt': FLF2V_PROMPT,
    'first_path': FLF2V_FIRST_IMAGE_PATH,
    'last_path': FLF2V_LAST_IMAGE_PATH,
}

'''
Keep the selected FLF2V job separate from the other modes.
讓 FLF2V 的檔案選取狀態與其他模式互不影響。
'''
flf2v_job = ACTIVE_JOB

'''
Save the uploaded image under its correct endpoint role.
依開頭或結尾用途儲存選取的圖片。
'''
def on_flf2v_upload(change, role):
    '''
    Update the reusable first and last image paths.
    更新可供下次使用的首尾圖片路徑。
    '''
    global FLF2V_FIRST_IMAGE_PATH, FLF2V_LAST_IMAGE_PATH
    selected = save_selected_image(change, 'flf_first' if role == 'first' else 'flf_last')
    if not selected:
        return
    if role == 'first':
        FLF2V_FIRST_IMAGE_PATH = selected
        flf2v_job['first_path'] = selected
    else:
        FLF2V_LAST_IMAGE_PATH = selected
        flf2v_job['last_path'] = selected
    print('開頭：', Path(flf2v_job['first_path']).name if flf2v_job['first_path'] else '未選取')
    print('結尾：', Path(flf2v_job['last_path']).name if flf2v_job['last_path'] else '未選取')

flf_first_upload = widgets.FileUpload(accept='.png,.jpg,.jpeg,.webp', multiple=False, description='選擇開頭照片')
flf_last_upload = widgets.FileUpload(accept='.png,.jpg,.jpeg,.webp', multiple=False, description='選擇結尾照片')
flf_first_upload.observe(lambda change: on_flf2v_upload(change, 'first'), names='value')
flf_last_upload.observe(lambda change: on_flf2v_upload(change, 'last'), names='value')

'''
Display both controls in this FLF2V mode cell.
在 FLF2V 模式 Cell 中同時顯示兩個上傳按鈕。
'''
display(flf_first_upload, flf_last_upload)
print('目前模式：FLF2V')
print('✓ FLF2V 模式設定完成：請確認首尾圖片已上傳，再執行 Generate。')


10. 生成影片


In [ ]:
'''
Serialize notebook generation with optional remote FIFO GPU jobs.
讓 Notebook 手動生成與選用的遠端 FIFO 共用 GPU 鎖，避免同時推論。
不必先啟動 ngrok 或任何 HTTP API。
'''
import threading
GENERATION_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
with GENERATION_LOCK:
    ltx_ensure_comfy_ready()
    spec = importlib.util.spec_from_file_location('ltx25_kaggle_engine', ENGINE_PATH)
    engine = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(engine)
    engine.RELEASE_AFTER_GENERATION = bool(RELEASE_AFTER_GENERATION)
    engine.STOP_AFTER_GENERATION = bool(STOP_COMFY_AFTER_GENERATION)
    engine.COMFY_PID = COMFY_PROC.pid if 'COMFY_PROC' in globals() and COMFY_PROC.poll() is None else None
    os.environ['LTX25_COMFY_RUNTIME_PID'] = str(engine.COMFY_PID or '')
    if 'ACTIVE_JOB' not in globals():
        raise ValueError('請先執行 T2V、I2V 或 FLF2V 的模式 Cell。')
    job = ACTIVE_JOB.copy()

    '''
    Read the selected mode and its independent prompt.
    讀取目前模式及其專屬 Prompt。
    '''
    MODE = job['mode']
    PROMPT = job['prompt']
    image_name = engine.ensure_placeholder_image()
    last_image_name = None
    if MODE in ('i2v', 'flf2v'):
        first_path = Path(job['first_path']) if job.get('first_path') else None
        if first_path is None or not first_path.is_file():
            raise ValueError(f'{MODE.upper()} 請先在模式 Cell 選取開頭照片。')
        RUN_NAME = job.get('run_name') or first_path.stem

        '''
        Upload the first frame to the ComfyUI input folder.
        將開頭照片放入 ComfyUI 的輸入資料夾。
        '''
        image_name, _ = engine.save_upload_bytes(first_path.read_bytes(), first_path.suffix, RUN_NAME)
    else:
        RUN_NAME = job['run_name']
    if MODE == 'flf2v':
        last_path = Path(job['last_path']) if job.get('last_path') else None
        if last_path is None or not last_path.is_file():
            raise ValueError('FLF2V 請在模式 Cell 選取結尾照片。')
        RUN_NAME = job.get('run_name') or f'{first_path.stem}_to_{last_path.stem}'

        '''
        Upload the final frame independently of the first frame.
        獨立上傳結尾照片，避免覆蓋開頭照片。
        '''
        last_image_name, _ = engine.save_upload_bytes(last_path.read_bytes(), last_path.suffix, f'{RUN_NAME}_last')
    import secrets

    '''
    Resolve one seed per run without overwriting the notebook setting.
    每次生成決定一個種子，但不覆寫 Notebook 的原始設定。
    '''
    RUN_SEED = secrets.randbelow(2**32) if SEED == -1 else int(SEED)
    assert 0 <= RUN_SEED < 2**32, 'SEED 必須介於 0 至 4294967295，或設為 -1。'
    print(f'本次生成 seed：{RUN_SEED}')
    OUTPUT_MP4 = ROOT / f'{RUN_NAME}.mp4'
    LAST_FRAME_PATH = ROOT / f'{RUN_NAME}_last_frame.png'
    print('MODE:', MODE, '| MP4:', OUTPUT_MP4.name)

    '''
    Preserve the audio batch-size setting required by the original workflow.
    保留原始 Workflow 所需的音訊 batch size 設定。
    '''
    queue_original = engine.queue_prompt

    '''
    Normalize audio batch size before submitting the prompt.
    送出 Prompt 前統一音訊的 batch size。
    '''
    def queue_with_audio_batch(workflow):
        '''
        Inspect the workflow for audio-latent nodes.
        尋找 Workflow 中的 audio latent 節點。
        '''
        for node in workflow.values():
            if node.get('class_type') == 'LTXVEmptyLatentAudio':
                node['inputs']['batch_size'] = 1
        return queue_original(workflow)
    engine.queue_prompt = queue_with_audio_batch
    params = {
        'mode': MODE,
        'prompt': PROMPT,
        'width': WIDTH,
        'height': HEIGHT,
        'seconds': SECONDS,
        'fps': FPS,
        'seed': RUN_SEED,
        'image_name': image_name,
        'last_image_name': last_image_name,
        'audio_name': engine.ensure_placeholder_audio(),
        'image_strength': IMAGE_STRENGTH,
        'output_policy': OUTPUT_POLICY,
        'vae_tile_size': VAE_TILE_SIZE,
        'vae_tile_overlap': VAE_TILE_OVERLAP,
        'vae_temporal_size': VAE_TEMPORAL_SIZE,
        'vae_temporal_overlap': VAE_TEMPORAL_OVERLAP,
    }
    available_gib = ltx_available_ram_gib()
    print(f'可用 RAM：約 {available_gib:.1f} GiB', flush=True)

    '''
    Reject high-risk jobs instead of allowing the OS to kill ComfyUI.
    可用 RAM 明顯不足時拒絕生成，避免讓作業系統直接終止 ComfyUI。
    '''
    if available_gib < LTX_MIN_AVAILABLE_RAM_GIB:
        raise RuntimeError(
            f'主機 RAM 不足：可用約 {available_gib:.1f} GiB，建議至少 {LTX_MIN_AVAILABLE_RAM_GIB:.0f} GiB。'
            '請先重新啟動 Kaggle Kernel、避免同時執行其他模型，或縮短影片秒數。'
        )
    start_time = time.perf_counter()
    try:
        generated_path = Path(engine.run_once(RUN_NAME, params, MODEL_TIER))
    except Exception:
        print('ComfyUI exit code:', COMFY_PROC.poll() if 'COMFY_PROC' in globals() else 'unknown')
        events_path = Path('/sys/fs/cgroup/memory.events')

        '''
        Show OOM kill evidence without assuming the process died from OOM.
        顯示 OOM Kill 訊號，不直接假定中斷原因。
        '''
        if events_path.is_file():
            print('Cgroup memory events:\n' + events_path.read_text())
        print(COMFY_LOG.read_text(errors='replace')[-4000:])
        raise
    shutil.copy2(generated_path, OUTPUT_MP4)
    if 'COMFY_PROC' in globals() and COMFY_PROC.poll() is not None:
        print('ComfyUI 已於本次影片完成後自動停止；下次生成前會自動重新啟動。')
    elapsed_seconds = time.perf_counter() - start_time
    print('MP4:', OUTPUT_MP4)
    print(f'生成耗時：{elapsed_seconds / 60:.2f} 分鐘（{elapsed_seconds:.1f} 秒）')

assert OUTPUT_MP4.is_file() and OUTPUT_MP4.stat().st_size > 0, '影片輸出不存在或為空。'
print('✓ 第 10 節完成：影片已生成並儲存 MP4 檔案。')


11. 預覽與下載


In [ ]:
'''
Confirm that the generated MP4 exists.
確認生成的 MP4 檔案存在。
'''
assert OUTPUT_MP4.is_file(), '請先執行生成影片 Cell。'
display(Video(str(OUTPUT_MP4), embed=True, width=448))
display(FileLink(str(OUTPUT_MP4)))
print('✓ 第 11 節完成：預覽及下載連結已顯示。')


12. 擷取最後影格（選用）


In [ ]:
'''
Verify output and FFmpeg tools before extracting the final I2V frame.
擷取 I2V 接續影格前，確認影片已完成且具備 FFmpeg 工具。
'''
assert OUTPUT_MP4.is_file(), '請先成功生成 MP4。'
assert shutil.which('ffmpeg') and shutil.which('ffprobe'), '找不到 ffmpeg／ffprobe；請先安裝。'
import json

'''
Read the frame count from the saved MP4.
從已儲存的 MP4 讀取影格總數。
'''
probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-count_frames', '-select_streams', 'v:0',
     '-show_entries', 'stream=nb_read_frames', '-of', 'json', str(OUTPUT_MP4)],
    capture_output=True, text=True, check=True,
)
last_index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
subprocess.run(
    ['ffmpeg', '-y', '-loglevel', 'error', '-i', str(OUTPUT_MP4),
     '-vf', rf'select=eq(n\,{last_index})', '-frames:v', '1', str(LAST_FRAME_PATH)],
    check=True,
)

'''
Preview and link the extracted image.
顯示擷取圖片並提供下載連結。
'''
display(Image.open(LAST_FRAME_PATH))
display(FileLink(str(LAST_FRAME_PATH)))
print(f"下一段：MODE = 'i2v'，IMAGE_PATH = '{LAST_FRAME_PATH}'")
print('✓ 第 12 節完成：最後影格已擷取並提供下載。')


13. （選用）啟動 ngrok 與 FIFO 遠端 API


In [ ]:
'''
Require the first configuration cell before enabling the optional remote API.
選用遠端 API 啟動前，必須先執行第一格共用參數設定。
'''
assert all(name in globals() for name in ('REMOTE_PORT', 'REMOTE_SERVICE_ID', 'REMOTE_ROOT')), '請先執行第 1 節共用參數設定。'
import zlib
import math
import re

import queue
import copy
import secrets
import io
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.request
import uuid
import zipfile
from email import policy
from email.parser import BytesParser
from getpass import getpass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlsplit, parse_qs

REMOTE_KIND = 'video'
REMOTE_FIELDS = ['task', 'prompt', 'width', 'height', 'seconds', 'fps', 'seed', 'image_strength', 'output_policy']
REMOTE_IS_LTX = True

'''
Provide safe defaults if the shared video-parameter Cell was not run in this Kernel.
即使本次 Kernel 尚未執行共用影片參數 Cell，也能提供相同的預設值。
'''
LTX_REMOTE_DEFAULTS = {
    'SECONDS': 10, 'FPS': 24, 'WIDTH': 288, 'HEIGHT': 512,
    'SEED': 42, 'IMAGE_STRENGTH': 0.7, 'MODEL_TIER': 'q4km',
    'OUTPUT_POLICY': 'exact_final',
    'VAE_TILE_SIZE': 256, 'VAE_TILE_OVERLAP': 32,
    'VAE_TEMPORAL_SIZE': 16, 'VAE_TEMPORAL_OVERLAP': 4,
}
required_setup = ('ROOT', 'ENGINE_PATH', 'COMFY_PORT', 'COMFY_LOG',
                  'ltx_available_ram_gib', 'LTX_MIN_AVAILABLE_RAM_GIB')

missing_setup = [name for name in required_setup if name not in globals()]
if missing_setup:
    raise RuntimeError('請先執行 Notebook 的環境、模型與 ComfyUI 啟動 Cells；缺少：' + ', '.join(missing_setup))

if not Path(ENGINE_PATH).is_file():
    raise RuntimeError(f'找不到 LTX 引擎：{ENGINE_PATH}；請重新執行模型與 Workflow Cell。')

REMOTE_SCRIPT = zlib.decompress(b'x\xda\xb5Z[W\x14I\xb6~\xe7W\xc4[\x15s\xa4\xb4\x19{f5\xab}\xa0\xb5\xa4k\x9a\xdb\x02tf\x8e\xc7\x95+\xa9\x8a\xa2\xb2M2\xebdf\x81\x8c\xc7\xb5\n\x97\x084\xa0t+\xd8*\xd3\xa8G[Z\xe5\xe2\x1d\x01\xf1\xc74\x99Y\xf5\xd4\x7f\xe1\xec\x1d;\xf2RI9=/\xc7\x07\x81\xa8\xc8\xc8}\xfd\xf6\xb7wT*\x95j\xe96\xd5\x02sJ\x9c\x99\x966\xa2\x19\xaa\xce\x9cq\xb3\xcdv\xd4\x11\xce\xba\x87\xfe\xd6\xd6\x9e\xf9\x9c\x8d\x9b\xd6\x85\xa2n\x8e3n\xc0\x16\x9ei\xa9\xed\xcd\xbaS\x8f\xdd\xeb\xab\xde\xcas\xff\xee\xd5pc\xfd\xde\x9d\xfa\xdd\xeb\xde\xc6\x1b\xf6\xd7\xe0\x11wo\xc9\xbby\xfd\xd7\xea\x95\x96\x14\xbcMw.*\xdc\xb0+\x16W\xf2\xe6hqB\xb1\xb8Z\x98H\xb7\xb6\xd8e\x9eg\'\x986Z6-G\xd7\x863\x15G\xd33\xb8\xaa\x14-sT)j:Wt3\xaf:\x9ai\xa4SpN\xfb\xe7\xca\x05ud\x04\x96I\xaa\xd4\x11\x96\xed\xed\xca\xf5f\x95\xfe\xce\xa1\xaf[[h\xf5\xf0\x99\xa3f\xa1\x02\x0f\x89S\xf1\xfc4\xfeG\x02dt0\x06\xb72\xfc"\xbc\x96\xf6\xa5\xe9\x98\xd6\x16!~g\xb9\xacO\x08k\xd9\xea(ge\xd3v\xda\xc6\xb4\x027Y^\xe7\xaaQ)3\xc7d\x16\x1f5\x1d\xceN\xe7N\xf7\xb1o\xcda\x9b\xa963`e\xd84/0\xabb\xd8`\xbf\x8d\x9b\xf5\xc9\xfb\xfe\xb3M\xda\xe5\xbe{|\xf0a\xa563\xcdz\x83}\x07\x1f>\xfa\xb7\xd6\xfc{\xdb\xee\xe2<X\xd8\xbf\xb5\xea\xcd,\xba\xfb\xf3\xde\xf6\x94\xbfx\xcd\xfbe\xd5\x9dy\x1b\x18Uze \xdb\x9d\xed\x1c\xcc*\x9d\xa7\x87\xb2\x03JW\xb67;\xd09\x94\xeb\xeb\x05\x13\xc0\x91z\xfaS\x9f\x07\xa6\xca\x0c\x0e\xf5\xf5\x7f\xf2i\xf1\xe1\xc9\xbe\x9e\xd3\x7for\x80ig\xb81\xa6Y\xa6q.\x05\xb1\x00\xae\xa1\x9d\x03gz\x87r=\xe0\x91\xdc\xa9\xd4y8\xc9v\xac4}\xd2?\xd0w2S\xd6\n\xe1\xcb\xe5r\xee\x14lk\xdc\xc2\xb4b\xc3\x8a\xa9\xeb\xe9V\xa6\xd9`,\xf0/\xd7m.~#\x0f\r@<1\xd5\x00\xa7\x8fV\x1cuX\x07O\x19j\xd9.\x99\x0e3\x8b\xc2s\xbaj;\x0c=\\qx\x81\x81\x979\xcbs]\xcf\xb4\xb87\x96\xdd\xfd\xdb\xdeJ\x15\xcc\xec\xae\xbe\xab=@\xc3{k\x0f\xdc\xbd\x1b\xec$\xec`\xb5\xb5uw\xe3\xeeo{\xf3\xf5\xc9\x8f\xee\xd4\x02\xf9\xe4`{\xbd^\xbdU{\xf8\xec`\xfb;\xf7\xca+\xef\xe6\x02\xb8\xd2\xbb\xf7\xda\x9d[\n\xfc\x03\xf2\xa7:O\x0e\xe5\xcef\x95\xbf\xf4}\x95\xc2``\x9a\xc1FtsX\xd5\xedtkGK\x0b\x83\x7f\xb8\x15\x7f\x0er\x9d\xe7A\\P.\x94\x8e\r\xf3\xa2i\x812\x8ej9\x9a1\xc2F\xb8\xc1-\x91\x0c\x19\xf1P}y\xce}2\'\xc3dv\x01\x84t?N\xd5\xef\xcf\xb8S3\xa4\x8b;\xf5\x16D=\xd8\xae\xba\xd5\xb9\x98R(c\xfc\xe5\x96\xaa\x81=\xcf\xaaz\x85g-\xcb\xb4\xd2\xa9\xda\xb3\xb9\xf0\x106\xd4~\xf6\xd7\xead\xae\xfd,\xf3f\x96\xd9\xe9\xee\xd3\xf0[\xa3\x95\xe0\xc0\x94\xcc\x96\x93f\x99\x92E\xcd;\xda\x18\xc7d\xc0\xfc([|\x8c\x1b\x0e8\xc2\xe1\x16\x1b\xd7\n#\x1c\x1c\x82K6\xc3\xacd\xf9\x92\x8a11\x02\xcf\x82\x97!c a\x1eM\xd7\xfe\xf7\x83\x7fo\x03t\x83\xf7\xc9\x84i\xf4\x07\xb8\xcd\xdf\x99\'7\x1c\xec\xcc\x1d\xec\xbe\xf5n\xbd\x077\x00Fy\xcf\x1f\x90i\x02\x97\xa0(\'X\xe4\x94L\x1edM\xb7\xc6bH$\xb9\xf0D\x10%\xaa\x01\xa1\x082jF\x81\x979\xfc\x07:\x94A\xde\xb2\x83\t]\x85\xf0!\xf9\xc8\x18\xee\x8d\xef\xc0\xe4\xee\xd6\x8c\xfb\xe29\xeb\x17\xdb\x82\x97\xf7\xf4\x9d\xca\xc2\xdbA\x86s)<9u\xbe\x05"\xbb\xa7\x7f(X\xa4SaYl\xff\x86\xf32SYYW\xf3\xbcd\xea\x00P\x10\xdd\x08\xcf\x10\x11\xcc\xe1\x17\x9d6\xd3\x00T\x8aG\x84\xff\xfa\x96\xb7<\xed\xae\xdf&Y\x08L\xea\xf7\x17\xc1b\xfe/;\xfe\x9d\x0f\xee\xca\xb2?;\x1d\x06(\x1e\xa7\x18\x08i\'\x02\x8c\x97(\x1d{\xab"\xb6\x81\x910\x83\x94\x86g\xe2\xf9\xf7\xdf\x15\r\x02UT\x14\xb0\x12\xba1\x92\x96k\xb0.\xc5o\xcb\x9bFACy\xa5\x811\x03\x9f\xd6~\x9e$iIB\xf2\x1ai\x811\xbd\xf5"\nk\x91\xae\x10\xf7\xf5\xfb\xeb\xfe\xd4\x93\xb8:E&L\x0c9\x96Ni\xedcP\x19RE\xbd\x08\xbf$S\xad\x0b"\xaf\xc1\xd1E\xcd\x02t \x81\xcb\xaaS\xa2\xe4\x92\xaf\x9a\xdcF\x1f\xdf\xbdZ{\xf3\x0er\x8d\x04\xac\xbd\xdbt\xf7\xbfOf\x918F\xc1\x03\xc08\xfd\xf0#-\xfc\x1a-\xa7\xce\xb7"\xac\xc1j\x06\xa2?\x1d\xff\xa45\x0ei\xf1C\x07\xf8\xb7\x88\x0b\xa3\x9am\x87V\xb5\x0f\x01CW\xff\x99C\xe0\xe0\xae\xac\x89\xf5\x04Hxs?\xf8o\x96\xfc\xbd\x1dw\xeb{inPN$UR!\x905\xa6S\x80\xbd\xe0Q\xc4\xb2\xe8\x93\x8cf\x8bJ\x1dbZ\xfc\x0c\xfc\x973\xa0\x04T\xf2d\xf5\x8a\r\xb1\x00\x80P)\xeb!\t\x91!S\x06\xc063\xe1c\xde\x8dE\xff\xd1\x0e\x05F\xad:\x05\xde\xa74\'OD\xdeo\xf6\xcaC\x88VL]\xc2\xf0\xc8T\xcaen\xa5[/3\x89p+kq\xb0\'o\'\xc2\x0b\x91-~\xfe)n!\xb0\xa1\xe4\x88\x89f\xc5)W\x1c&\x92B \x99\x88-\xb3b\xe59\xb9\xeb\x08\xab\x18:\xb7\x81\x12\x04,\x01\x81H\xec\x07\x9bB\xda#\x99(\x90\xde\x02dg\xe7\x00\xc0d\xa1\x11\xda\x1f\xec_\xf3v\x16\xbd\xa7\xb7\xdc\xc5\x05\x81|\xc8"\xe2\x1b\xfc\x855\x1f\xf4\x11\x00\t{\xfc\xb5\x17I_BQVz;{$\x04Q\xfc\x01\xc8\x8a\\\x86\xe8\x03\xa7\xc6\x1cj;|\xb4Q\xe73\x91\xb7([\x8a\x16\xca\x0f~\xc4\xa5\x93\xc8\xea\xce\xe4 \xf7\xd0\x12E\x01\x1c2\x08\xb7\xa6\xe3\xd6\xf4n\xed\x03\x85\x0c\xf7c\xe4\xedm\xc3J\xed\xd5\xb4\xb7|\xc7}\xb4\x7f(\x04C\xc09\xc2\x94\x08\xa7lu\x8c+\x14A\xca\xf0\x84\xc3\xedtLzd\x97r\xb5\xf5H\x83Z\x95bQ\xbbx$\xb4\x05\xd0\x10H\xba\x04:\x9c\xb1\xc9\xb5\xfcbY\xd7\xf2\x9a\x83\xd5\x0f\x88\x9eV&\x8f\x8d\x97\xb8\x01\xf1/\xd1\x02\xfcgCD\xe9Z\xe0?\xdc\xec\xbd\xfc\xc1[\x99%\x9fQ\x86\x81\xcfd\x18O?\xad\xad]\xc1Z\xf6\xe1\x05\xac\xff;\xae:\x17\xb9\xe9|#\xda\xaa\xac\xa0\xd9\x90\xfe\x90XP\x98\x1a1Wh}\x141\xbb\x8d<\x15G\x87\xfa\xcf\xcb\xee\xd6>\x88\xe0\xdd\xdf\xa3\xe8\xaf\xafT\xeb\x0f\x16\xdcG\xcb\xde\xfc4\x00\x81\xfff\x1164\xc7\xd7\x13\'\x02\\\xedH"\x95\x0c\x90\x06aB\x90\x1d\x96\x14\x9ac\x05\x08\xb2\x7f\xb8\xe28\x01^\xc9j\xfa\xfc\xf9\xc1\xce<\xd2\x95\xf9\xd9\xfa\xccR\x08\xbfI\x91\xe2o\x16\xa5\xe9\x10\xee\x86\xabI\xd8\x8d>\xf8=\xd4\x05J\t\xcc\x1d\xca\x05X4@\xe0\xa2h\x97\x84r$7\xb9\x9b\x0c\x07\xfc\xc8\x9b\xdd?\xd8^pg\xb6\xe2\x12{w\xae`~.\xbc\xf2\xe6\xe6\xdc\xe9\x9d\xfa\xfcf\xed\xe9\xa3&p\x1bi\x92@\xdb\xf0\x83\xdf\x05\xdb,\xc4\xad\nEp\xbc\xa4\xe5K2#\xe10)~\x12]\xa9\x0e43\xee\xef"j\x8a\x88 \xe2h3\x10M\x9c\x99\x04\xd1^\x01\x1f\x10\x10\xa3\xd0A\x81AE\x86U\x84\x85\x87M\xd4\x9fB\x194\xc5\xd0\xb7\x8f\xa0!. \'\xc2g\x120JJ\x918\x87\xc1\xd3\x9d\xfa\xc5\xdd{\x13\x96\xba\xffo\x14M]J\x00\xe9e\xc51\x95K\x91\x03\xc5Z\xea_\xc0+\xc6\x17%m\x8ct\x02\xd7\x93\xadL\x0c\x7fIq\x12\x9bjc\xdc\xea!G\xae\xfd|\xadvs.I\x9a\x0ee\xd0\xbf\x8d\xb5\x91.\x8dP\x1b\xd3Q"-X#0\xd9e\x05?\x0e\xba\x84.\x82#\x04\xb1\xa2\xc5\xed\x12\xfbc{\xdb0\x80\xad\xcd\x01*\x04\xb3\x15@\xdb\xf6\x19\xc6\xae\x05\x80\xc7\xc1l\x05A\x16!\xd9\x88\xae\xe0\xa7\x90X\xde\xec\x82\xbb\xbb\x03\x16\xf0\x96\xb7p,!O\xaa\xdf]\x83\x86\xd9_\xdbp\xd7\x17#\xca\x8bC\x01\xc4 \x8b;v\x80\xa6\xb6\xa9C9\xc7d\x13o\x07~\x80\xed\x07\xf4(N\tj;3\xc7\xb85ni\x0e\xf5&<\xea\xea\x01\x18\x1c\x91R\xde\xe6\x8d\xb0\xd7\xf0^\xec\x80h\xd4q\xd1\xcb\x05\xf8_\x03L\x007\xb8\x9b\xcf\xa2n\x1f\xcb\xc0\xf5U\xa02\x11\xfb\x12\x02\x81\xc1\x06\xb3Yl\x89\xa5\xa0\x19\x0b\xba\x90a\xae\x9b\xe3\xe9\xf6?\xfc\xe1\x8f\xed\x02\xceh\xcb\t4\x82\x001\xcdp\xd2\xb8\xd6\x9a,\x12X\xa0@k\xd0L\xa8\'\xa9\xe3pE\xd3\x0b\x81J\x9aQ\xe4\x167\x80\xb0\x04C\x1f\xb0\xf4\xca\x9a4\xeb\xf5\xb5\xda\xb3e\xca\n\xef\xcd\xa4\xbf6\'[\xcd\xa9\x19\xef\xe9Co\xf51\xa9\xe9o^uW\x16\x02%T\x1bh\x9e\xc3\x8e\xb1/O\xb0P\xa1/\x99\x10\x1f\x88\xb9\xf8\x93H\xfd\xc1.8n\x0fv\xd6\xa6_\xb1\xe3\xed_\x1c\xff\xe2O\x7fn\xff\xe2sx\x01\x80)\xb6.Wv@GD\x10Rl\xb0d\x8e\x07\xfd%\x02\x87P\xca6\x91\\\x81\x95\x80~\xa1\xef\xf2\x80\xdc\xc3\x88\x13\xc0\xae\n\x95<\xec\x10}\'\x94\xbe\x07\x9b\x80z\xee\xe6\xcf\xf5\xbb\xf7d\xd2Cm\t\xfc\xe4-\xbf?\xd8\xffx\xf0~\x0e\xba\xca\xfa\xf4\x82\x7fc_F\x91p-\xe4\x96\xf7\xd3J\xa0a\xd9B\x8b\x17S\xf1FS\x08\xf3\xdb\xde\xddK\x81\xca\x97\xa3\xae8\xe0\xc0\x92.\x06S-`\x85E\xec\x88\x9bw\x9e!\xfe1\x15|v\xc10\xc7\xb1\xcf{\x88!\x166\x9c\x04h 1\xfa\xe4\x9a\xcc\x05\xa4U\xd3;\xc42j3\xd3T\xeb\x91n\x85\x9d\x0bJ\xd5wf\xa8\xff\xcc\x90\xd2\xd3\x7f\x1c\x82m\xa0\xafo\x88\x1dmH\xd9\xcch\xf9x\xaa\xa5\xbbspH9=\xd0\xd9CC\xb6\xe6[Ev+\x84Jec$\xe9\xac|\xc9\xb4!\x9f\xe3\\Y\x06\xa2\xe4Z\xd0y"\xa9\xc6\x86\x05\xda\x81\x80\xa9@\x10\xd6\xaa\xb7\xb1zBb\x8bffr\xce\xdf\xb8\r\x01H\x8e\xacW\xef\xa0\xf1\x89FN\xef\x90)\x1a=\x94B\xd6\xd2\x01\xdd \xfe\x84\xd0\xfb\x1f\x06\xea\xe2\xdf\x91\xf2\x19\x94G:\xaa\x1f\xd0\x88[\x92\xde\xab\x95\x82f\xb2a\xd5\xc9\x97\xdal\xed\x1f<Hy\x81H\x90]!\xab\t\xc7\xa8Q\xfe\x1c|\xfc\xa7\xbft\x87\x12<\x1a\x8cz\xb3U\xe0Z p}\xf5Um\xed;:\x9b\x89\xb3\x1bQ\x00\x10\xaf\xc2\x95\xf0\xdc\x10\x8di\x9df\x05$q\xafi\x8d\xaa:\x9e\x10\x13\x97\x8e\x0c\xda\xc4\xca\xf0\xa8\xe6\x84\x00\x16\x8c/\xea\xd5I\xb0\x99\x1cS0\x9c\xb1\xbcy\x01\xc0E\x92\xa1\xc1\xa3\x93\x02\xa9\n\xbc\xc8H\x02DGE\xbcP\x11\xdb\xd2\x81\xe6\xc9n\x1b\x9a\xbf2\x97\xbd_8RF\x8a*\x1en\xc3\xbc4 " \xe2\xed\xa0e\x98\x03\x0e\x15\x99\xec`{\x1d\x85!\xe5\xe4v\x7f\xb3Z\xdf\xfd\xe9P\x03.\xc8R\x01\xf1,r\xc4\x182\x16\xfb\x93t\x89\xe6,I\xc3\xe5\xc1$\x10\x828IDE\x85\xf04\xf2\xa5\x08\x8eh\x14$#x\xba\x897E\xd2a\xf6\x89\xc8\xc4\x99\xee\xc6\xfbOq+\x80s\x94\x9b\xf8D\x1e2\xc9V\x9c\x8922\nd\xdb\xddC\x7f;\x9b\x05\x17Mt\x0b\xe5;Q\xd8TL\x9d\xe4iA\x17#*\xda\x84\x91/Y\xa6\x01"\x15\xa4\x9a6VS\xc0\xfaL\xc3\x03\xb2\xd1\xdc\xae\xfao\xae\xba\x8b\xf3\xde\xfac\xd2).r\xb3\x17\xa1\xdc\xe7R\x82f\xda\xa9\xf3\xe7R\xc2\x04\n\x9a@\xcc~?KL9E$6O\x998\x10B@\xeb\xa1\x134]s&\x98Z\xf8\xb6b;\xa3`\x00\x19&\x1b\xf3\x88\xc9\xc2\xae^\xf5I\xed\xe9Go\xe95\xa1 \x05v"\xf5\x0e\xcd<\xb9S\xb1\x0c\xd6\x98gQ\x14\'G\xffd\xbbO\x89D\x8dp0\xb7\xc4\xa6\x97\x92(!\xdd\xe3\xdb\xa2\x07|\xd5l8\xd9$\xbf\xc1~\xcd\x93MB\x15\xc4Ic\xdd\x90\xf0d\x07}y\x13X\x02\xd9\x08h\xdc\xea\x1e\xf0\xc5\x86\x0b\x9d\xb8\xa9\x04\x80\xaa\x00\xe76Hq\x89,\'\xc6\x96\x1d\x04\xa4\xb4"g\x96\x1d\x8cF\x99ru\\+@s\xd5\xc1\xfe\x9a;5\xf4\xb5\\+qm\xa4\x84;\xbf\xce\xe6\xba\xbe\x0evR7h\xc3\xf2`\xf6d_\xef\xa9A\xb9^,\xe3\xda\xe9\xfe\xc1p\x1f/\xc0BPV\xe5jDX\xe1\xb3\x18{\xa5O\x13\x9c\x16\xb6$Y.\xed#\xbb\xca-\x9f\x9e\x83\x8am\xc0r\xe3\xaf\x86\x92\x0e\x0f\x08]s=\x9d]Yeph \xdb\xdb\x15*M`\xa1\x94M]\xcbO\xc0&Yr\xfa\xfb\xbas\'\xff.\xf7\x8c\xa9\\q\xf0\x9eK\xe4L\x07;\xdb\x99U\x86r\xddpV\xee?\xb3\xc9=\xc8Eu\xb5\x1c\xdf\xd6w6;\xd0\xdd\xd9\x1f\xdf\xc9\x91\xe6\xaaz\xc3\x89\xd9\x9e\xfe\xbe\x81\xce\xeeC\xa7\x06{\x13\'\x07\xdb\xc3\xd3/K\x1eS\xe2\xf9\x0b\x0c\x8a\xb9#\xf8I~\xc42+e\x06\xdc\xe0\xd0H\x12\x18XT\xd6\t;\xa3(\xa4\x99$\x94\xf6\xda\xcb]\x9c\xfd\xec,\x863\x91\xd8%\xa2\xa0\x99\xc41\x0f\xb6w\x81\x87!\xa8n\xbcw\xef\xac\x11\x81so@\xfb\xb6\x86o\x0fi\xe7\x98\xaa\xe9x\x01\xa4\x8ch8\xeb\xc7\x9b\xc7h\r\xa2\x19\xd7\xd3\xad!y\x8bN\x00\xd6\xe6\xbf\xbe\xca.5\x9c\xd0\x91\xf9\xacx\x99ui_\x01](\xea\x15\xbbtb\xc8\xaa\xf0\x90a\x8b\x19A\t\xe2\xba\xcd\xd2\xec\x0bt\xed\xa7\x01\xcf\xc3A\x084k\x80a\xe6xPt\xfb\x061)/h\x80kr\xfa\x85\xcdL\xf0v\xe6\xfdx\x1d\x18\r6\to_aOCCZ\x01\x10Q\x1b\xb7q\x13)\xf8\xe3u\xff\xd5.\x94j\xff\xdek\xef\xfac\xff\xcd\x15o\xfdapflX\xd3h\x8a/\xf1\xb6V\xe9\xc9\xf5*\x9dg;s\xdd\x9d_A\xe4\xc0{\x95\xae\xdcW\x1d\xb1+\xa0\x81\x8a\xe1h\xa3\xb2\xc1\x0fq\xbe\x98"\xf3\x0bIIF0\x17I\xff/\x8c\x86|tw\xa7\xb6\xbe\x05\x88\xe7n}\xcf.}Z\x84\xcc1zD\xd0\xfc\xb0\xd4\xd0\x88\x16\x888tv\xee\xd2*\xf4\xf5\xec\x1bq\xf9\x0bU\xdb2\xb8\xfekuR^\x02A\xc1\xbas%\xba\xf2\xda]Fv\xfc\xd3\x1c5\x11\xfe\xf6\x86\xbf\xbaN\xd1\xe5?\xf9\xc1[\xda\x0e_#]\x19+K\xd1\x9cL\x04\xf8(W\x11\t\x88\xc3\x02\xd6\xe9\xdc\xc1\xdbG\xb5l\x03\xdc\xa2\xa9\xc0\x8btk(\xef\x05\x7f\xf6\xb6\x9f\xd7\xa7o\xf8\xdb\x1f\x88\xb8\x06\x1e\x11y\xa1\xe0\x13\x10\x96\xe2A`\xb9E%o\x82\xc5q -c\nz\x97\x86WE\xc5\x11\x84\xb1\x91M\xab\xf8\x03Z+\x8e>\x826b\x04\xd9\x03\x08_\x04\x17\xd8\x814X\x1c\x97^\x87\xa9\x84\x9ex\xf4\xc2[B\x81\x888\xfb/\xf7\xfd\xad\x074\x87\xf2_W\xeb\xf3W\x03A\x1dk\xe2\xd0}\x89\xec\xd5\x1bj\x8d\x18\xd8\x08r\x84\xd7c\x86*.\xfb\xa2o\x11D\xb9\x1e\xb1\x0b\xa84`\xa6&_\x17\x90\x97\x13\xc2C\xc9:-\x1d\xc2\x0b\r\x13>\x89\xd38{1\x81\xcb\xa4\x83>\xe4\x08\xa3\xa2Et\xbf\x1b 2;\xd0\xda\xda\xc2/\xe69\xd4\xd3\xac\xf8\x01\xbe=4\xba\x14s\x01\xd2\x0f\xc8\xbf\x05\xd0\x05&\xcd#\x97\x14s\x08M\xcc\xd0\x0c\xe0\x9f\xa61B\xf3\x01#\x9c\xa6\x05\xd3l\xf7\xddK\xe0\xaa\xde\xf2\xbb\xd0\xca\x18\x17ks\xd87\xfes\xd5\x7f\xb8\x97TM6(\xc1\xf3\xe1;\xb1;ir\xf7]d\xa9h5\xd5p\x99L#\x80T\xc5\x10-br\xd6F\xa0\xdd\xad\x19\x95\x8b\x10\xd0\xa3\xa65\x81\xd7p\x8e\x05\xe7\x82.2\x04m\xd2\x14l0!\x9a\xcd0\xa7\xe5\x90I`\x8ewk\xd3\xbb\xf1\x1a\xf4\xa3a\xad<\xb3\xb6\xf6\xa3w\xedm\xfd\xd9-\xef\xfa\x13w\xe6-\xe0smm\x06\xd2\x0c~I\xaaL7\xbe\r\xaeL\x1d\xb5\'\xec\xa3E\xfb(\x15\x93\xa3$a\x86v&U\x11\re__\x0f!)\xc7\xba"\xc6\x16rV\x03\x8c\xa82\x1a\xebs\xf2x\xedR\xd0\xf0\x8e\x0fof\xe0Ayq.\xdc#\x0e\xfaF\x13\x97\xfc\xdf\xd5\xee\xac\xe2\xa4f{\x81p\xd5\x9d\x9c\x16s\x1c\xf4\'2\xc9{\xf7\x9b\x8ckc\xda\xfc\xee\\\xb6\x1f\x9d-\x93[\x1a\x1c\n\x02M\xf0G*\x9a]\x92\xae\x11\x9a\xc9;qS\\\x9cbf\x03\x06\xd9\x11a\x97\xd1\x15\x18\x19\xe5\xde}\xec.T\xdd\x99k\xa1/\xa80`\xc5\x14\x80\xd88rN\n\x17D"\x95s)\x08i\xd7\xf1_F\x8a\xfdG\x83\xaab\xea\x87w\xd0\xe9\xd6\x84{"%-\xc0\x891\xd5\x10W\x13\xc1\xe42\x08t\x99c\x80]qox+\xd5\xda\xc7\xef\x91\x87\x06\xfb\xbc\xdb\x8fkO\x01\xd0\x9f\xc4Z\xa7\xc6\xdc\xa1|\xe8\xee\xeb\x8a\xc9$\x80\xd1>\x91\xb2\xb8\xa0o\xa9\xd6sm\xc7\x8f\x1d;\xd6q\xfe\x90\xac\xb1N?d\xca\x04\xab\x88\xb6\x8e\x05O\x0f\xab\xf9\x0b\x12\xc2b\xfd<\x19\x13\xbf\xba\x13nj\xfa\x95\x8a\xc47"\xe0\x85\x15]\x90#\x9c\xb8\x80\xfc\xc4\xd2m\xfa\xca\x8a\x9c\x8d\xd0\x054\xd0\xf48(\xbaW_\xba\xeb?\xba3[\xee=\x9c\xfdPO\xd98\xcb\xb1K\xe2\x8bN\xf8\x95\x86\xf6t#h\xc6g\x1d!y\t\x11/1\x90"\xddM\'\xde\x99Q\xa1\x93\xb3\xb3XW{\xf0a\xc1\xff\xb0!X\x19\xd6\x1b\x92\xb8\xb1\xee\xc9Z\xa9H\xa6\xdf\xbc\xf8\xb16\x16\xd5\xc7pn\x93\x1c\xd3D\xb4-\xfe"\x1c\xb7%_r\x94\xfd\xe9XG\xa6\x1d8\x05dD}\xf1\xc7\xdf\xf6f\x92{\x88\xa5\x00\x1d\xf8mo\x16@\xe6\xff\x00\x98\xea\x1c\xf4').decode('utf-8')
REMOTE_ALLOWED_TASKS = ['t2v', 'i2v', 'flf2v']
REMOTE_MAX_IMAGES = 2

REMOTE_ROOT.mkdir(parents=True, exist_ok=True)
REMOTE_JOBS = globals().setdefault('REMOTE_JOBS', {})
REMOTE_JOB_LOCK = globals().setdefault('REMOTE_JOB_LOCK', threading.Lock())
REMOTE_ENGINE_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
REMOTE_QUEUE = globals().setdefault('REMOTE_QUEUE', queue.Queue(maxsize=REMOTE_QUEUE_CAPACITY))
assert REMOTE_QUEUE.maxsize == REMOTE_QUEUE_CAPACITY, 'FIFO 容量已變更，請先重新啟動 Kernel 及遠端服務。'
REMOTE_UPLOAD_SLOTS = globals().setdefault('REMOTE_UPLOAD_SLOTS', threading.BoundedSemaphore(REMOTE_UPLOAD_CONCURRENCY))

'''
Results remain available for up to 24 hours, subject to an 8 GiB storage limit.
結果最多保留 24 小時；若超過 8 GiB，會提前清除最舊結果。
'''
REMOTE_RESULT_TTL = REMOTE_RESULT_TTL_HOURS * 3600
REMOTE_RESULT_BYTES = REMOTE_RESULT_LIMIT_GIB * 1024**3
REMOTE_MIN_FREE_BYTES = REMOTE_MIN_FREE_GIB * 1024**3
REMOTE_STALL_SECONDS = REMOTE_STALL_HOURS * 3600  # 無限大代表不因等待時間長而拒絕工作

'''
Verify the CUDA backend and existing model without loading another model.
確認 CUDA 與已啟動的模型後端，不另外載入第二份模型。
'''
def remote_gpu_status():
    try:
        import torch
        if not torch.cuda.is_available():
            return False, 'CUDA GPU 不可用；請檢查 NVIDIA 驅動與 PyTorch，並重新執行模型初始化 Cell。', []
        qwen = REMOTE_KIND == 'image_multi'
        yue = REMOTE_SERVICE_ID.startswith('06_YuE2_')
        if yue:
            selected_gpus = [globals().get('GPU_ID', 0)]
        elif qwen and globals().get('QWEN21_STATE', {}).get('mode') != 'bf16_dual':
            selected_gpus = [globals().get('GPU_ID', 0)]
        else:
            selected_gpus = sorted({
                globals().get('MODEL_GPU_ID', 0),
                globals().get('TEXT_GPU_ID', 1) if REMOTE_SERVICE_ID.startswith(('06_MiniMax_', '06_HyperFlow_'))
                else globals().get('AUX_GPU_ID', 1) if REMOTE_SERVICE_ID.startswith(('06_HeartMuLa_', '06_LTX-2.5_', '06_Z-Image-Turbo_'))
                else globals().get('TEXT_GPU_ID', 1) if qwen else 0
            }) if not qwen else sorted({globals().get('GPU_ID', 0), globals().get('TEXT_GPU_ID', 1)})
        available = torch.cuda.device_count()
        if any(not isinstance(index, int) or index < 0 or index >= available for index in selected_gpus):
            return False, f'需要 cuda:{selected_gpus}，但實際只有 {available} 張 CUDA GPU。', []
        devices = [f'cuda:{index} {torch.cuda.get_device_name(index)}' for index in selected_gpus]
        if qwen:
            state = globals().get('QWEN21_STATE', {})
            if state.get('pipeline') is None or state.get('manager') is None:
                return False, 'Qwen Pipeline／GPU 記憶體管理器尚未初始化，請先執行模型初始化 Cell。', devices
            if state.get('device') != f"cuda:{globals().get('GPU_ID', 0)}":
                return False, 'Qwen Transformer 不在設定的 GPU 上。', devices
        elif REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
            pipeline = globals().get('pipe')
            if pipeline is None or pipeline.lazy_load or pipeline._mula is None or pipeline._codec is None:
                return False, 'HeartMuLa 雙 GPU Pipeline 尚未完整載入，請先執行模型載入 Cell。', devices
            for name, module, index in (('HeartMuLa', pipeline.mula, globals().get('MODEL_GPU_ID', 0)), ('HeartCodec', pipeline.codec, globals().get('AUX_GPU_ID', 1))):
                parameter = next(module.parameters(), None)
                if parameter is None or parameter.device != torch.device(f'cuda:{index}'):
                    return False, f'{name} 未位於指定的 cuda:{index}，請重新載入模型。', devices
        elif yue:
            gpu_id = globals().get('GPU_ID', 0)
            process = globals().get('SERVER_PROCS', {}).get(gpu_id)
            if process is None or process.poll() is not None:
                return False, f'YuE2 CUDA{gpu_id} 常駐服務尚未啟動。', devices
            if f'CUDA{gpu_id}' not in [str(value) for value in process.args]:
                return False, 'YuE2 常駐服務沒有使用預期的 CUDA 啟動參數。', devices
        elif REMOTE_IS_LTX and STOP_COMFY_AFTER_GENERATION:
            return True, 'CUDA 已就緒；ComfyUI 在每次生成前自動啟動。', devices
        else:
            base = globals().get('BASE')
            if not base and REMOTE_IS_LTX:
                base = f"http://127.0.0.1:{globals().get('COMFY_PORT', 8188)}"
            if not isinstance(base, str) or not base:
                return False, 'ComfyUI 尚未啟動，請先執行模型服務 Cell。', devices
            with urllib.request.urlopen(base.rstrip('/') + '/system_stats', timeout=4) as response:
                stats = json.load(response)
            comfy_devices = stats.get('devices', [])
            if not any(str(item.get('type', '')).lower().startswith('cuda') or
                       str(item.get('name', '')).lower().startswith('cuda:')
                       for item in comfy_devices if isinstance(item, dict)):
                return False, 'ComfyUI 沒有回報 CUDA 裝置，已拒絕 CPU-only 推論。', devices
        return True, 'CUDA 後端已就緒（不代表所有權重常駐 GPU 記憶體）。', devices
    except Exception as error:
        return False, f'GPU 檢查失敗：{type(error).__name__}: {error}', []

'''
Verify every global required by the selected Notebook generation engine.
驗證所選模型的生成程式所需變數與設定已完成初始化。
'''
def remote_inference_ready(task):
    namespace = globals()
    common = {
        '06_HeartMuLa_': ('ROOT', 'pipe'),
        '06_YuE2_': ('ROOT', 'GPU_ID', 'PORTS', 'SERVER_PROCS'),
        '06_Z-Image-Turbo_': ('ROOT', 'BASE', 'LOG_PATH'),
        '06_Qwen_Image_': ('MODEL_MODE', 'GPU0_MODEL_GIB', 'GPU_ID', 'QWEN21_STATE'),
        '06_LTX-2.5_': ('ROOT', 'ENGINE_PATH', 'COMFY_PORT', 'COMFY_LOG', 'COMFY_PROC',
                          'ltx_available_ram_gib', 'LTX_MIN_AVAILABLE_RAM_GIB', 'importlib'),
        '06_MiniMax_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                              'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                              'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                              'QUANT_SIZES_GIB', 'ready', 'choices', 'check_choice'),
        '06_HyperFlow_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                                'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                                'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                                'QUANT_SIZES_GB', 'ready', 'choices', 'ENCODERS',
                                'ENCODER_FILE', 'MODEL_FAMILY', 'WEIGHTS_IN',
                                'TEXT_LOADER_NODE', 'PROJECTION_FILE'),
    }
    required = next((names for prefix, names in common.items()
                     if REMOTE_SERVICE_ID.startswith(prefix)), ())
    missing = [name for name in required if name not in namespace or namespace[name] is None]
    if missing:
        return False, 'Notebook 尚未完成初始化，請先執行對應的模型／服務 Cells；缺少：' + ', '.join(missing)
    if task not in REMOTE_ALLOWED_TASKS:
        return False, '此模型不支援生成模式：' + task
    if REMOTE_SERVICE_ID.startswith(('06_MiniMax_H3_', '06_HyperFlow_H3_')):
        config = namespace['TASK_CONFIGS'].get(task)
        if not isinstance(config, dict):
            return False, f'{task.upper()} 尚未設定；請先執行 Notebook 對應的模式設定 Cell。'
        if namespace['H3_ACTIVE_CONFIG'] != namespace['H3_DESIRED_CONFIG']:
            return False, 'GPU／Offload 設定與已啟動 ComfyUI 不一致，請重新啟動模型服務 Cell。'
        quant = config.get('quant')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            variant = 'ref2va_pruned' if task == 'r2v' else 'fl2va_pruned'
            expected = (variant, quant)
            expected_file = f'minimax_h3_{variant}-{quant}.gguf'
        else:
            variant = 'ref2va' if task == 'r2v' else 'fl2va'
            encoder = config.get('encoder_variant')
            expected = (variant, quant, encoder)
            expected_file = f'minimax_h3_{variant}_hyperflow_flash_14b-{quant}.gguf'
            if encoder not in namespace['ENCODERS']:
                return False, f'{task.upper()} 的 Encoder 設定無效，請重新執行模式設定 Cell。'
            if Path(namespace['ENCODER_FILE']).name != namespace['ENCODERS'][encoder][1]:
                return False, 'GPU 1 Encoder 與此模式不一致，請重新執行模型下載／啟動 Cells。'
        if namespace['H3_MODEL_CHOICE'] != expected or namespace['H3_ACTIVE_MODEL'] != expected:
            return False, f'{task.upper()} 尚未載入相應模型權重；請設定模式並重新執行模型下載／啟動 Cells。'
        if namespace['DIT_NAME'] != expected_file:
            return False, 'ComfyUI 使用的模型檔名與任務不一致，請重新載入正確模型。'
    elif REMOTE_SERVICE_ID.startswith('06_Qwen_Image_'):
        state = namespace['QWEN21_STATE']
        if state.get('mode') != namespace['MODEL_MODE']:
            return False, 'Qwen MODEL_MODE 與已初始化的模型模式不同，請重新執行模型初始化 Cell。'
        if state.get('gpu0_model_gib') != namespace['GPU0_MODEL_GIB']:
            return False, 'Qwen GPU0_MODEL_GIB 與目前載入設定不同，請重新初始化 GPU 記憶體管理器。'
    elif REMOTE_IS_LTX:
        if not Path(namespace['ENGINE_PATH']).is_file():
            return False, 'LTX 模型引擎檔案不存在；請先執行模型與 Workflow Cell。'
        if namespace['COMFY_PROC'].poll() is not None and not namespace.get('STOP_COMFY_AFTER_GENERATION', False):
            return False, 'LTX ComfyUI 行程已停止，請重新執行啟動 ComfyUI Cell。'
    return True, '模型初始化與所選模式已就緒。'

'''
Validate client parameters before they occupy a FIFO slot.
在進入 FIFO 前檢查 client 參數，避免排隊後才因設定錯誤而失敗。
'''
def remote_validate_fields(task, fields):
    required = {'task'}
    if REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
        required.update(('lyrics', 'tags', 'duration', 'seed'))
    elif REMOTE_SERVICE_ID.startswith('06_YuE2_'):
        required.update(('style', 'lyrics', 'duration', 'steps', 'lm_seed', 'seed', 'cot'))
    elif REMOTE_KIND == 'image_text':
        required.update(('prompt', 'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_KIND == 'image_multi':
        required.update(('prompt', 'negative_prompt', 'sampler', 'reference_size',
                         'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_IS_LTX:
        required.update(('prompt', 'width', 'height', 'seconds', 'fps', 'seed', 'image_strength'))
    else:
        required.update(('prompt', 'width', 'height', 'seconds', 'fps', 'steps', 'seed', 'ref_image_size'))
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            required.add('audio_shift')
    missing = required - fields.keys()
    if missing:
        raise ValueError('請求缺少必要參數：' + ', '.join(sorted(missing)))
    integer_fields = ('seed', 'lm_seed', 'width', 'height', 'steps', 'fps', 'duration', 'reference_size')
    float_fields = ('seconds', 'cfg', 'image_strength', 'audio_shift')
    numbers = {}
    for key in integer_fields:
        if key in fields:
            raw = fields[key]
            if isinstance(raw, bool) or str(raw).strip() != str(int(raw)):
                raise ValueError(f'{key} 必須是整數')
            numbers[key] = int(raw)
    for key in float_fields:
        if key in fields:
            numbers[key] = float(fields[key])
            if not math.isfinite(numbers[key]):
                raise ValueError(f'{key} 必須是有限數值')
    for key in ('seed', 'lm_seed'):
        if key in numbers and not -1 <= numbers[key] < 2**32:
            raise ValueError(f'{key} 必須介於 -1 至 4294967295')
    for key in ('width', 'height'):
        if key in numbers:
            size = numbers[key]
            divisor = 64 if REMOTE_KIND == 'image_text' else 32
            if size < 1 or size > 4096 or size % divisor:
                raise ValueError(f'{key} 必須介於 1～4096 且為 {divisor} 的倍數')
    for key in ('seconds', 'duration'):
        if key in numbers and not 0 < numbers[key] <= 3600:
            raise ValueError(f'{key} 必須大於 0 且不超過 3600')
    '''
    Reject unsupported FPS values before placing jobs in the GPU queue.
    在工作排入 GPU 佇列前，先拒絕不支援的 FPS 設定。
    '''
    if REMOTE_IS_LTX:
        for key in ('width', 'height'):
            if numbers[key] < 256:
                raise ValueError(f'LTX {key} 至少須 256 px；此欄位代表要求交付的最終尺寸')
        if fields.get('output_policy', 'exact_final') not in ('exact_final', 'native'):
            raise ValueError('output_policy 只能是 exact_final 或 native')
    if 'fps' in numbers:
        if REMOTE_IS_LTX and not 1 <= numbers['fps'] <= 60:
            raise ValueError('LTX-2.5 FPS 必須介於 1～60')
        if not REMOTE_IS_LTX and numbers['fps'] != 24:
            raise ValueError('MiniMax H3／HyperFlow H3 僅支援 24 FPS')
    if 'steps' in numbers and not 1 <= numbers['steps'] <= 200:
        raise ValueError('steps 必須介於 1～200')
    if REMOTE_SERVICE_ID.startswith('06_HyperFlow_H3_') and numbers.get('steps') not in (4, 6, 8):
        raise ValueError('HyperFlow steps 只支援 4、6、8')
    if 'image_strength' in numbers and not 0 <= numbers['image_strength'] <= 1:
        raise ValueError('image_strength 必須介於 0～1')
    if 'reference_size' in numbers and not 64 <= numbers['reference_size'] <= 2048:
        raise ValueError('reference_size 必須介於 64～2048')
    if 'audio_shift' in numbers and not 0.01 <= numbers['audio_shift'] <= 100:
        raise ValueError('audio_shift 必須介於 0.01～100')
    if fields.get('ref_image_size', 'match') not in ('match', 'max'):
        raise ValueError('ref_image_size 只支援 match 或 max')
    if REMOTE_KIND in ('video', 'image_text', 'image_multi') and not str(fields.get('prompt', '')).strip():
        raise ValueError('PROMPT 不可為空白')
    if task == 'r2v':
        matches = [int(number) for number in re.findall(r'<Picture\s+(\d+)>', str(fields['prompt']), re.I)]
        if any(number < 1 or number > int(fields.get('_image_count', 0)) for number in matches):
            raise ValueError('R2V Prompt 的 <Picture N> 編號超出已上傳的圖片數量')

'''
Report whether the model is ready for CUDA inference.
回報模型是否已具備 CUDA 推論條件。
'''
def remote_backend_ready():
    return remote_gpu_status()[0]

'''
A Python thread cannot safely terminate an already running CUDA inference.
Python thread 無法安全強制終止正在執行的 CUDA 推論。
'''
def remote_stalled():
    with REMOTE_JOB_LOCK:
        return any(record.get('status') == 'running' and
                   time.monotonic() - record.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
                   for record in REMOTE_JOBS.values())

'''
Remove obsolete input files once the model no longer needs them.
模型不再需要輸入檔案時立即移除，避免參考圖片長期佔用空間。
'''
def remote_remove_inputs(job_id):
    folder = REMOTE_ROOT / job_id
    for file in folder.glob('*'):
        if file.name.startswith('input.') or (file.stem.isdecimal() and file.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}):
            file.unlink(missing_ok=True)

'''
Keep only finished results; never delete active jobs or in-progress downloads.
僅清理已結束工作，絕不刪除排隊、推論或正在下載的檔案。
'''
def remote_cleanup():
    now = time.time()
    with REMOTE_JOB_LOCK:
        finished = [(job_id, record) for job_id, record in REMOTE_JOBS.items()
                    if record.get('status') in ('done', 'error', 'expired') and
                    not record.get('_downloads')]
        finished.sort(key=lambda item: item[1].get('finished_at', now))
        sizes = {}
        for job_id, record in finished:
            folder = REMOTE_ROOT / job_id
            sizes[job_id] = sum(path.stat().st_size for path in folder.glob('**/*') if path.is_file()) if folder.exists() else 0
        total = sum(sizes.values())
        for job_id, record in finished:
            old = now - record.get('finished_at', now) > REMOTE_RESULT_TTL
            if old or total > REMOTE_RESULT_BYTES:
                shutil.rmtree(REMOTE_ROOT / job_id, ignore_errors=True)
                total -= sizes[job_id]
                record.update(status='expired', error='結果已過期或因儲存空間不足而清除，請重新提交。')
                record.pop('_path', None)
                record.pop('_last_frame', None)
            if record['status'] == 'expired' and now - record.get('finished_at', now) > 2 * REMOTE_RESULT_TTL:
                REMOTE_JOBS.pop(job_id, None)

'''
Sweep completed records periodically without interfering with inference.
定期整理已完成資料，不干擾單一 GPU worker 的推論。
'''
def remote_maintenance():
    while True:
        try:
            remote_cleanup()
        except Exception as error:
            print('API cleanup warning:', type(error).__name__, str(error)[:200])
        time.sleep(300)

'''
Serve only the generation API without exposing notebook controls.
只提供生成 API，不公開 Notebook 的管理功能。
'''
class RemoteHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        return

    def send_json(self, code, payload, retry_after=None):
        data = json.dumps(payload, ensure_ascii=False).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(data)))
        if retry_after is not None:
            self.send_header('Retry-After', str(retry_after))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        path = urlsplit(self.path).path
        if path == '/health':
            with REMOTE_JOB_LOCK:
                running = sum(record.get('status') == 'running' for record in REMOTE_JOBS.values())
            gpu_ready, gpu_message, gpu_devices = remote_gpu_status()
            requested_task = parse_qs(urlsplit(self.path).query).get('task', [REMOTE_ALLOWED_TASKS[0]])[-1]
            inference_ready, inference_message = remote_inference_ready(requested_task)
            if gpu_ready and not inference_ready:
                gpu_message = inference_message
            ram_gib = ltx_available_ram_gib() if REMOTE_IS_LTX else None
            ram_ready = ram_gib is None or ram_gib >= LTX_MIN_AVAILABLE_RAM_GIB
            if not ram_ready:
                gpu_message = (f'主機 RAM 不足：可用約 {ram_gib:.1f} GiB，'
                               f'建議至少 {LTX_MIN_AVAILABLE_RAM_GIB:.0f} GiB；'
                               '請重新啟動 Kaggle Kernel 或降低生成負載。')
            self.send_json(200, {'ok': True, 'api': 'kaggle-generation-bridge-v3',
                                 'service_id': REMOTE_SERVICE_ID, 'task': requested_task,
                                 'backend_ready': gpu_ready and inference_ready and ram_ready, 'gpu_ready': gpu_ready,
                                 'gpu_message': gpu_message, 'gpu_devices': gpu_devices,
                                 'ram_ready': ram_ready, 'ram_available_gib': round(ram_gib, 2) if ram_gib is not None else None,
                                 'stalled': remote_stalled(), 'running': running,
                                 'waiting': REMOTE_QUEUE.qsize(), 'queue_capacity': REMOTE_QUEUE.maxsize})
            return
        if path == '/schema':
            self.send_json(200, {'kind': REMOTE_KIND, 'tasks': REMOTE_ALLOWED_TASKS,
                                 'fields': REMOTE_FIELDS, 'image_limit': REMOTE_MAX_IMAGES,
                                 'upload_field': 'images' if REMOTE_MAX_IMAGES else None,
                                 'queue_capacity': REMOTE_QUEUE.maxsize,
                                 'model_switching': 'configure in notebook first',
                                 'media_contract': {'dimension_semantics': 'final_pixels',
                                     'output_policy_default': 'exact_final',
                                     'output_policy_options': ['exact_final', 'native'],
                                     'min_dimension': 256, 'dimension_multiple': 32,
                                     'native_spatial_scale': '1 or 2 depending on verified workflow',
                                     'duration_tolerance_seconds': 'max(0.1, 2/fps)'} if REMOTE_IS_LTX else None})
            return
        pieces = path.strip('/').split('/')
        if pieces[0] != 'jobs' or len(pieces) not in (2, 3):
            self.send_json(404, {'error': 'Not found'})
            return
        job_id = pieces[1]
        with REMOTE_JOB_LOCK:
            record = REMOTE_JOBS.get(job_id)
            item = dict(record) if record else None
            if item and item['status'] == 'running':
                item['stalled'] = time.monotonic() - item.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
            if item and item['status'] == 'queued':
                waiting = [key for key, value in REMOTE_JOBS.items() if value.get('status') == 'queued']
                if job_id in waiting:
                    item['queue_position'] = waiting.index(job_id) + 1
            if item and len(pieces) == 3 and pieces[2] in ('file', 'last-frame') and item['status'] == 'done':
                record['_downloads'] = record.get('_downloads', 0) + 1
                downloading = True
            else:
                downloading = False
        if item is None:
            self.send_json(404, {'error': 'Unknown job'})
            return
        if len(pieces) == 2:
            for field in tuple(item):
                if field.startswith('_'):
                    item.pop(field)
            self.send_json(200, item)
            return
        if not downloading:
            self.send_json(404, {'error': 'Result not available for this job'})
            return
        try:
            file_path = item.get('_path') if pieces[2] == 'file' else item.get('_last_frame')
            if not file_path or not Path(file_path).is_file():
                self.send_json(404, {'error': 'Output is no longer available'})
                return
            result = Path(file_path)
            self.send_response(200)
            self.send_header('Content-Type', 'application/octet-stream')
            self.send_header('Content-Disposition', f'attachment; filename="{result.name}"')
            self.send_header('Content-Length', str(result.stat().st_size))
            self.end_headers()
            with result.open('rb') as stream:
                shutil.copyfileobj(stream, self.wfile, length=1024 * 1024)
        except (BrokenPipeError, ConnectionResetError):
            pass
        finally:
            with REMOTE_JOB_LOCK:
                record['_downloads'] -= 1

    def do_POST(self):
        if urlsplit(self.path).path != '/generate':
            self.send_json(404, {'error': 'Not found'})
            return
        supplied_id = self.headers.get('X-Request-ID', '').strip()
        try:
            job_id = uuid.UUID(supplied_id).hex if supplied_id else uuid.uuid4().hex
        except ValueError:
            self.send_json(400, {'error': 'X-Request-ID must be a UUID'})
            return
        with REMOTE_JOB_LOCK:
            existing = REMOTE_JOBS.get(job_id)
        if existing:
            self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                 'download_url': f'/jobs/{job_id}/file', 'reused': True})
            return
        if remote_stalled():
            self.send_json(503, {'error': 'GPU worker appears stalled; inspect the Notebook before accepting more jobs'}, 60)
            return
        gpu_ready, gpu_message, _ = remote_gpu_status()
        if not gpu_ready:
            self.send_json(409, {'error': gpu_message})
            return

        '''
        Reject new video jobs if host RAM is already critically constrained.
        主機 RAM 已不足時拒絕新影片工作，避免繼續累積記憶體壓力。
        '''
        if REMOTE_IS_LTX and ltx_available_ram_gib() < LTX_MIN_AVAILABLE_RAM_GIB:
            self.send_json(503, {'error': 'Kaggle host RAM is too low for LTX-2.5 inference'}, 30)
            return
        if shutil.disk_usage(REMOTE_ROOT).free < REMOTE_MIN_FREE_BYTES:
            self.send_json(507, {'error': 'Kaggle disk space is low; clean old outputs before retrying'})
            return
        if not REMOTE_UPLOAD_SLOTS.acquire(timeout=1):
            self.send_json(503, {'error': 'Upload slots busy; retry later'}, 5)
            return
        staging = None
        try:
            self.connection.settimeout(120)
            length = int(self.headers.get('Content-Length', '0'))
            if not 0 < length <= 120 * 1024 * 1024:
                raise ValueError('Request must be between 1 byte and 120 MiB')
            body = self.rfile.read(length)
            if len(body) != length:
                raise ValueError('Incomplete request body')
            content_type = self.headers.get('Content-Type', '')
            fields, files = {}, []
            if content_type.startswith('application/json'):
                fields = json.loads(body)
            elif content_type.startswith('application/x-www-form-urlencoded'):
                fields = {key: values[-1] for key, values in parse_qs(body.decode('utf-8')).items()}
            elif content_type.startswith('multipart/form-data'):
                message = BytesParser(policy=policy.default).parsebytes(
                    b'Content-Type: ' + content_type.encode() + b'\r\nMIME-Version: 1.0\r\n\r\n' + body)
                for part in message.iter_parts():
                    name = part.get_param('name', header='content-disposition')
                    data = part.get_payload(decode=True)
                    if name == 'images':
                        files.append((part.get_filename() or 'upload.png', data))
                    elif name:
                        fields[name] = data.decode('utf-8')
            else:
                raise ValueError('Use application/json, form data, or multipart/form-data')
            if not isinstance(fields, dict):
                raise ValueError('Parameters must be a JSON object')
            if 'abc' in fields and len(str(fields['abc']).encode('utf-8')) > 1024 * 1024:
                raise ValueError('ABC score must be 1 MiB or smaller')
            task = str(fields.get('task', REMOTE_ALLOWED_TASKS[0])).lower()
            if task not in REMOTE_ALLOWED_TASKS:
                raise ValueError('Unsupported task: ' + task)
            if len(files) > REMOTE_MAX_IMAGES:
                raise ValueError(f'Only {REMOTE_MAX_IMAGES} input images are supported')
            if REMOTE_KIND in ('music', 'image_text') and files:
                raise ValueError('This model does not accept reference images')
            if REMOTE_KIND == 'video':
                minimum, maximum = {'t2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2),
                                    'r2v': (1, REMOTE_MAX_IMAGES)}[task] if task == 'r2v' else {
                                    't2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError('Input image count does not match the selected video task')
            if REMOTE_KIND == 'image_multi':
                minimum, maximum = {'text': (0, 0), 'edit': (1, 1), 'multi': (2, REMOTE_MAX_IMAGES)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError(f'Task {task} requires {minimum}–{maximum} images')
            for key in fields:
                if key not in REMOTE_FIELDS:
                    raise ValueError('Unsupported parameter: ' + key)
            remote_validate_fields(task, dict(fields, _image_count=len(files)))
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                self.send_json(409, {'error': inference_message})
                return
            for filename, data in files:
                if not isinstance(data, bytes) or len(data) > 20 * 1024 * 1024:
                    raise ValueError('Each image must be 20 MiB or smaller')
                if Path(filename).suffix.lower() not in {'.png', '.jpg', '.jpeg', '.webp'}:
                    raise ValueError('Only PNG/JPEG/WebP images are supported')
                from PIL import Image
                with Image.open(io.BytesIO(data)) as photo:
                    photo.verify()
            staging = Path(tempfile.mkdtemp(prefix='pending_', dir=REMOTE_ROOT))
            image_names = []
            for index, (filename, data) in enumerate(files, 1):
                target = staging / f'{index:03d}{Path(filename).suffix.lower()}'
                target.write_bytes(data)
                image_names.append(target.name)
            with REMOTE_JOB_LOCK:
                existing = REMOTE_JOBS.get(job_id)
                full = REMOTE_QUEUE.full()
                if not existing and not full:
                    folder = REMOTE_ROOT / job_id
                    staging.rename(folder)
                    staging = None
                    image_paths = [str(folder / name) for name in image_names]
                    REMOTE_JOBS[job_id] = {'id': job_id, 'task': task, 'status': 'queued',
                                           'created_at': time.time()}
                    REMOTE_QUEUE.put_nowait((job_id, fields, image_paths))
                    position = REMOTE_QUEUE.qsize()
            if existing:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'reused': True})
            elif full:
                self.send_json(429, {'error': 'Queue is full; retry later'}, 15)
            else:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'queue_position': position})
        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:
            self.send_json(400, {'error': str(error)})
        except (TimeoutError, ConnectionError, OSError) as error:
            self.send_json(503, {'error': f'Upload temporarily failed: {type(error).__name__}'}, 10)
        except Exception as error:
            self.send_json(500, {'error': type(error).__name__ + ': ' + str(error)[:250]})
        finally:
            if staging is not None:
                shutil.rmtree(staging, ignore_errors=True)
            REMOTE_UPLOAD_SLOTS.release()

'''
Convert numeric HTTP form fields into explicitly supported generator settings.
只轉換白名單中的 HTTP 參數，不執行客戶端提供的任何程式碼。
'''
def remote_generate(job_id, fields, image_paths):
    with REMOTE_ENGINE_LOCK:
        with REMOTE_JOB_LOCK:
            REMOTE_JOBS[job_id].update(status='running', _start_clock=time.monotonic())
        namespace = globals()
        keys = ['TASK', 'PROMPT', 'WIDTH', 'HEIGHT', 'SEED', 'LM_SEED', 'SECONDS', 'STEPS', 'CFG',
                'LYRICS', 'TAGS', 'STYLE', 'DURATION', 'GPU_ID', 'COT', 'ABC_PATH',
                'OUTPUT_NAME', 'REFERENCE_PATHS', 'ACTIVE_JOB', 'TASK_CONFIGS',
                'R2V_REF_DIR', 'NEGATIVE_PROMPT', 'SAMPLER', 'REFERENCE_SIZE',
                'OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT', 'RUN_SEED',
                'OUTPUT_POLICY',
                 'FPS', 'IMAGE_STRENGTH', 'MODEL_TIER', 'VAE_TILE_SIZE',
                 'VAE_TILE_OVERLAP', 'VAE_TEMPORAL_SIZE', 'VAE_TEMPORAL_OVERLAP']
        before = {key: copy.deepcopy(namespace.get(key)) for key in keys if key in namespace}
        missing = set(keys) - set(before)
        try:
            '''
            Fill missing LTX runtime constants without overwriting notebook settings.
            補齊缺少的 LTX 執行參數，但不覆蓋 Notebook 已設定的值。
            '''
            if REMOTE_IS_LTX:
                for name, value in LTX_REMOTE_DEFAULTS.items():
                    namespace.setdefault(name, value)

            task = fields.get('task', REMOTE_ALLOWED_TASKS[0]).lower()
            numerics = {'seed': int, 'lm_seed': int, 'width': int, 'height': int, 'steps': int,
                        'seconds': float, 'fps': int, 'cfg': float, 'duration': int,
                        'image_strength': float, 'reference_size': int, 'audio_shift': float}
            inputs = {}
            for key, value in fields.items():
                inputs[key] = numerics[key](value) if key in numerics else value
            for seed_key in ('seed', 'lm_seed'):
                if seed_key in inputs:
                    if not -1 <= inputs[seed_key] < 2**32:
                        raise ValueError(f'{seed_key} must be -1 or between 0 and 4294967295')
                    if inputs[seed_key] == -1:
                        inputs[seed_key] = secrets.randbelow(2**32)
            with REMOTE_JOB_LOCK:
                for seed_key in ('seed', 'lm_seed'):
                    if seed_key in inputs:
                        REMOTE_JOBS[job_id][seed_key + '_used'] = inputs[seed_key]
            for dimension in ('width', 'height'):
                if dimension in inputs and not 0 < inputs[dimension] <= 4096:
                    raise ValueError('Width and height must be between 1 and 4096')
            for key in ('seconds', 'duration'):
                if key in inputs and not 0 < inputs[key] <= 3600:
                    raise ValueError('Duration must be between 0 and 3600 seconds')
            if 'reference_size' in inputs and not 64 <= inputs['reference_size'] <= 2048:
                raise ValueError('Reference size must be between 64 and 2048')
            if 'ref_image_size' in inputs and inputs['ref_image_size'] not in ('match', 'max'):
                raise ValueError('Reference image mode must be match or max')
            if 'image_strength' in inputs and not 0 <= inputs['image_strength'] <= 1:
                raise ValueError('Image strength must be between 0 and 1')
            if REMOTE_IS_LTX:
                namespace['OUTPUT_POLICY'] = inputs.get('output_policy', 'exact_final')
            if 'steps' in inputs and not 1 <= inputs['steps'] <= 200:
                raise ValueError('Steps must be between 1 and 200')
            if REMOTE_KIND == 'video':
                if 'TASK_CONFIGS' in namespace:
                    if task not in namespace['TASK_CONFIGS']:
                        raise ValueError('Run this task configuration cell before API generation')
                    updated = copy.deepcopy(namespace['TASK_CONFIGS'][task])
                    for field in ('prompt', 'width', 'height', 'seconds', 'steps', 'seed',
                                  'audio_shift', 'ref_image_size'):
                        if field in inputs:
                            updated[field] = inputs[field]
                    updated['output_name'] = f'remote_{job_id[:12]}.mp4'
                    if task == 'i2v':
                        updated['first_image'] = image_paths[0]
                    elif task == 'flf2v':
                        updated['first_image'], updated['last_image'] = image_paths
                    elif task == 'r2v':
                        updated['ref_images'] = image_paths
                        namespace['R2V_REF_DIR'] = Path(image_paths[0]).parent
                    namespace['TASK_CONFIGS'][task] = updated
                    namespace['TASK'] = task
                else:
                    namespace['ACTIVE_JOB'] = {'mode': task, 'prompt': inputs.get('prompt', namespace.get('PROMPT', '')),
                                               'run_name': f'remote_{job_id[:12]}',
                                               'first_path': image_paths[0] if image_paths else None,
                                               'last_path': image_paths[1] if len(image_paths) > 1 else None}
                    for key in ('width', 'height', 'seconds', 'fps', 'seed', 'image_strength'):
                        if key in inputs:
                            namespace[key.upper()] = inputs[key]
            else:
                for key in ('prompt', 'width', 'height', 'seed', 'lm_seed', 'steps', 'cfg',
                            'lyrics', 'tags', 'style', 'duration', 'negative_prompt',
                            'sampler', 'reference_size', 'cot'):
                    if key in inputs:
                        namespace[key.upper()] = inputs[key]
                if REMOTE_KIND == 'image_multi':
                    namespace['TASK'] = task
                    namespace['REFERENCE_PATHS'] = image_paths
                if REMOTE_KIND == 'music':
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.mp3'
                    if 'ABC_PATH' in namespace:
                        namespace['ABC_PATH'] = ''
                    if 'abc' in inputs:
                        score = (REMOTE_ROOT / job_id / 'input.abc')
                        score.write_text(str(inputs['abc']), encoding='utf-8')
                        namespace['ABC_PATH'] = str(score)
                if REMOTE_KIND in ('image_multi', 'image_text'):
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.png'
            for output_key in ('OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT'):
                namespace.pop(output_key, None)
            '''
            Recheck the CUDA backend after queuing to prevent CPU-only generation.
            工作排入佇列後再次確認 CUDA，避免後端異常時改走純 CPU 生成。
            '''
            gpu_ready, gpu_message, _ = remote_gpu_status()
            if not gpu_ready:
                raise RuntimeError(gpu_message)
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                raise RuntimeError(inference_message)
            if REMOTE_IS_LTX:
                available_gib = ltx_available_ram_gib()
                if available_gib < LTX_MIN_AVAILABLE_RAM_GIB:
                    raise RuntimeError(f'生成前主機 RAM 不足：{available_gib:.1f} GiB 可用，'
                                       f'至少需要 {LTX_MIN_AVAILABLE_RAM_GIB:.0f} GiB 緩衝。')
            exec(compile(REMOTE_SCRIPT, '<notebook-generation-cell>', 'exec'), namespace)
            media_contract = None
            if REMOTE_IS_LTX:
                engine_module = namespace.get('engine')
                if engine_module is not None:
                    media_contract = engine_module.JOBS.get(namespace.get('RUN_NAME'), {}).get('media_contract')
            paths = []
            if REMOTE_KIND == 'video':
                paths = [namespace.get('LAST_OUTPUT') or namespace.get('OUTPUT_MP4')]
            elif REMOTE_KIND == 'image_text':
                paths = [namespace.get('OUTPUT_PATH')]
            elif REMOTE_KIND == 'image_multi':
                paths = [namespace.get('output')]
            elif REMOTE_KIND == 'music':
                paths = namespace.get('OUTPUTS', []) if 'OUTPUTS' in namespace else [namespace.get('output')]
            results = [Path(path) for path in paths if path is not None]
            if REMOTE_KIND == 'music' and REMOTE_SERVICE_ID.startswith('06_YuE2_'):
                results += [sidecar for path in list(results)
                            for ext in ('.json', '.abc')
                            if (sidecar := path.with_suffix(ext)).is_file()]
            if not results or any(not path.is_file() for path in results):
                raise FileNotFoundError('Generation finished without a saved result file')
            folder = REMOTE_ROOT / job_id
            if len(results) == 1:
                target = folder / f'result{results[0].suffix.lower()}'
                shutil.copy2(results[0], target)
            else:
                target = folder / 'results.zip'
                with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_STORED) as archive:
                    for result in results:
                        archive.write(result, arcname=result.name)
            last_frame = None
            if REMOTE_KIND == 'video' and shutil.which('ffprobe') and shutil.which('ffmpeg'):
                try:
                    probe = subprocess.run(['ffprobe', '-v', 'error', '-count_frames',
                                            '-select_streams', 'v:0', '-show_entries',
                                            'stream=nb_read_frames', '-of', 'json', str(target)],
                                           check=True, text=True, capture_output=True, timeout=120)
                    index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
                    if index >= 0:
                        last_frame = folder / 'last_frame.png'
                        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(target),
                                        '-vf', rf'select=eq(n\,{index})', '-frames:v', '1',
                                        str(last_frame)], check=True)
                except (OSError, ValueError, KeyError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
                    last_frame = None
            for original in results:
                if original.name.startswith(f'remote_{job_id[:12]}') and original != target:
                    try:
                        original.unlink(missing_ok=True)
                    except OSError:
                        pass
            comfy = globals().get('COMFY')
            if comfy is not None and (Path(comfy) / 'output').exists():
                for artifact in (Path(comfy) / 'output').rglob(f'remote_{job_id[:12]}*'):
                    if artifact.is_file():
                        try:
                            artifact.unlink()
                        except OSError:
                            pass
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='done', filename=target.name,
                                           bytes=target.stat().st_size, _path=str(target),
                                           media_contract=media_contract if REMOTE_IS_LTX else None,
                                           last_frame_available=bool(last_frame),
                                           _last_frame=str(last_frame) if last_frame else None)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            for key in missing:
                namespace.pop(key, None)
            namespace.update(before)

'''
Use one GPU worker, regardless of the number of HTTP request threads.
不論有多少 HTTP 連線，都只有一個 GPU worker 消費佇列。
'''
def remote_worker():
    while True:
        job_id, fields, image_paths = REMOTE_QUEUE.get()
        try:
            remote_generate(job_id, fields, image_paths)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id]['finished_at'] = time.time()
            try:
                remote_remove_inputs(job_id)
            except OSError as error:
                print('API input cleanup warning:', str(error)[:180])
            finally:
                REMOTE_QUEUE.task_done()

if not (globals().get('REMOTE_WORKER') and REMOTE_WORKER.is_alive()):
    REMOTE_WORKER = threading.Thread(target=remote_worker, daemon=True)
    REMOTE_WORKER.start()

if not (globals().get('REMOTE_MAINTENANCE') and REMOTE_MAINTENANCE.is_alive()):
    REMOTE_MAINTENANCE = threading.Thread(target=remote_maintenance, daemon=True)
    REMOTE_MAINTENANCE.start()

old_tunnel = globals().get('REMOTE_NGROK_PROC')
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()
        old_tunnel.wait(timeout=5)

NGROK_URL = ''

'''
Install the official ngrok agent only if no usable binary exists.
系統尚無 ngrok 才下載官方執行檔，不重複安裝。
'''
ngrok_bin = shutil.which('ngrok') or str(ROOT / 'ngrok')
if not Path(ngrok_bin).is_file():
    url = 'https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz'
    with urllib.request.urlopen(url, timeout=90) as response:
        blob = response.read()
    with tarfile.open(fileobj=io.BytesIO(blob), mode='r:gz') as archive:
        entry = next((member for member in archive.getmembers() if Path(member.name).name == 'ngrok' and member.isfile()), None)
        if entry is None:
            raise RuntimeError('ngrok 官方下載檔案缺少執行檔')
        Path(ngrok_bin).write_bytes(archive.extractfile(entry).read())
    Path(ngrok_bin).chmod(0o755)

ngrok_token = getpass('輸入 ngrok Authtoken：').strip()
if not ngrok_token:
    raise ValueError('未輸入 ngrok Authtoken')

result = subprocess.run([ngrok_bin, 'config', 'add-authtoken', ngrok_token], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok token 設定失敗（請確認 Token 是否有效）')

del ngrok_token
result = subprocess.run([ngrok_bin, 'config', 'check'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok 設定格式檢查失敗：' + result.stderr[-500:])

server = globals().get('REMOTE_SERVER')
server_thread = globals().get('REMOTE_SERVER_THREAD')
server_is_current = (
    server is not None
    and server_thread is not None
    and server_thread.is_alive()
    and getattr(server, 'notebook_id', None) == REMOTE_SERVICE_ID
)
if server is not None and not server_is_current:
    if server_thread is not None and server_thread.is_alive():
        server.shutdown()
    server.server_close()

if not server_is_current:
    try:
        REMOTE_SERVER = ThreadingHTTPServer(('127.0.0.1', REMOTE_PORT), RemoteHandler)
    except OSError as error:
        raise RuntimeError(f'本機 API Port {REMOTE_PORT} 已被其他服務占用') from error
    REMOTE_SERVER.notebook_id = REMOTE_SERVICE_ID
    REMOTE_SERVER.daemon_threads = True
    REMOTE_SERVER_THREAD = threading.Thread(target=REMOTE_SERVER.serve_forever, daemon=True)
    REMOTE_SERVER_THREAD.start()
    print('已啟動新的本機 API Server。')
else:
    REMOTE_SERVER.RequestHandlerClass = RemoteHandler
    print('已沿用本機 API Server；現有生成工作與結果保持不變。')

with urllib.request.urlopen(f'http://127.0.0.1:{REMOTE_PORT}/health', timeout=4) as response:
    if json.load(response).get('ok') is not True:
        raise RuntimeError('本機 API Server 尚未就緒')

'''
Use the agent inspection API to obtain the actual public URL.
透過 ngrok 本機檢查 API 取得實際對外 HTTPS 網址。
'''
NGROK_LOG = REMOTE_ROOT / 'ngrok.log'
with NGROK_LOG.open('w') as log:
    REMOTE_NGROK_PROC = subprocess.Popen([ngrok_bin, 'http', str(REMOTE_PORT)], stdout=log,
                                         stderr=subprocess.STDOUT, start_new_session=True)

NGROK_URL = ''
for attempt in range(25):
    if REMOTE_NGROK_PROC.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=2) as response:
            tunnels = json.load(response).get('tunnels', [])
        NGROK_URL = next((item['public_url'] for item in tunnels
                          if item.get('public_url', '').startswith('https://') and
                          str(REMOTE_PORT) in item.get('config', {}).get('addr', '')), '')
        if NGROK_URL:
            break
    except (OSError, ValueError, KeyError):
        pass
    time.sleep(1)

if not NGROK_URL:
    raise RuntimeError('ngrok URL 建立失敗：\n' + NGROK_LOG.read_text(errors='replace')[-2500:])

print('ngrok HTTPS URL:', NGROK_URL)
print('提醒：公開的 ngrok API URL 不需要額外驗證，請只分享給授課學生。')
print('GET /health | GET /schema | POST /generate | GET /jobs/{id} | GET /jobs/{id}/file')
print('影片輸出另提供：GET /jobs/{id}/last-frame')
print('✓ 第 13 節完成：選用 FIFO／ngrok API 已啟動。')


14. （選用）遠端 API 與 client 使用說明


In [ ]:
'''
Query the active bridge for supported parameters and task names.
從正在執行的橋接服務讀取實際支援的參數與模式。
'''
import json
import urllib.request
request = urllib.request.Request(f'http://127.0.0.1:{REMOTE_PORT}/schema')
with urllib.request.urlopen(request, timeout=5) as response:
    schema = json.load(response)

print(json.dumps(schema, ensure_ascii=False, indent=2))
print('用法：POST NGROK_URL + "/generate"（JSON 或 multipart/images），取得 job_id；')
print('輪詢 GET /jobs/{job_id}；完成後下載 GET /jobs/{job_id}/file。')
print('本機 requests 範例：06_LTX-2.5_影片生成_client.py（與本 Notebook 同名對應）。')
print('使用前只需在對應的 client.py 填入 API_URL。')
print('本服務使用單一 FIFO worker，遠端請求不會同時啟動多筆 GPU 推論。')
print('✓ 第 14 節完成：API Schema 和 client 說明已顯示。')
